
---
# Proyecto: Data Mining
# **Segmentación de distritos del Perú según movilidad escolar en primaria y secundaria (2023-2024)**

**1. Formulación definida**

**Problema:** Cada año, algunos estudiantes no vuelven a matricularse en ninguna escuela, lo que interrumpe su educación
básica. Este problema no se reparte por igual: el promedio
nacional parece estable, pero oculta distritos donde la salida
es mucho mayor, y hoy no es claro dónde se concentra ni si
pesa más en primaria o en secundaria.

**Importancia:** Cada estudiante que sale del sistema pierde oportunidades de aprendizaje y de desarrollo futuro, y los costos recaen sobre el distrito y el país. Si la salida se mide solo en promedios, los distritos más afectados quedan invisibles y los recursos de retención (becas, transporte, tutorías, seguimiento de estudiantes en riesgo) no llegan a donde más se necesitan. Una segmentación distrital permite pasar de un diagnóstico general a una priorización concreta.

**Usuario:** Ministerio de Educación del Perú (MINEDU), Direcciones Regionales de Educación (DRE), Unidades de Gestión Educativa Local (UGEL) y analistas de políticas públicas educativas.

**Objetivo:** Integrar cuatro fuentes públicas del sector Educación (Perú) en una sola base
de análisis a nivel **distrito x nivel educativo**, usando el **Padrón de Instituciones
Educativas (`Padron.dbf`)** como tabla puente entre la matrícula (a nivel de escuela) y la
deserción (a nivel de distrito/ubigeo).

**Análisis:** Comparar las tasas de salida de estudiantes entre primaria y secundaria a escala distrital e identificar qué territorios presentan las mayores brechas.

**Fuentes:**
1. `Padron.dbf` — catálogo de IIEE. Puente: da la ubicación geográfica (`ubigeo`/`CODGEO`) de cada
   escuela (`COD_MOD` + `ANEXO`).
2. `Matriculación_y_Trayectoria_Estudiantil_2023.csv` — matrícula y trayectoria por escuela y nivel, año 2023. Se filtra a **Primaria** y **Secundaria**.
3. `Tasa_y_número_de_desertores_en_Educación_Primaria_2023-2024.csv` — nivel Primaria.
4. `Tasa_y_número_de_desertores_en_Educación_Secundaria_2023-2024.csv` — nivel Secundaria.

**Definición importante de las fuentes 3 y 4 (deserción):** no son un conteo independiente por
año; es una **medida de cohorte**: `denominador` = estudiantes matriculados en **2023**, y
`desertor` = de ese grupo, cuántos **no volvieron a matricularse en 2024** (en cualquier escuela
del sistema). `Tasa` = `desertor / denominador`. Esto implica que la deserción debe cruzarse
contra la **matrícula 2023** (que es su propio denominador).

**Unidad de análisis:** Distrito por nivel educativo (o distrito-nivel). Cada registro representa el comportamiento educativo de un distrito específico en un nivel formativo determinado (Primaria o Secundaria).

**Pregunta:**  ¿Qué perfiles de distritos se distinguen según su salida escolar y dónde están los de mayor salida en cada nivel?

---


# **2. Fuentes**

---

# <font color=green>Parte 1. Cargar e inspeccionar las fuentes</font>
Antes de integrar, debemos entender qué contiene cada fuente.

In [ ]:
# @title
!pip install dbfread -q

In [ ]:
# @title
!pip install dbfread pandas
import pandas as pd
import numpy as np
import plotly.express as px
from dbfread import DBF


RUTA = "/content/"


### <font color=279CF5>Preparación: guardado automático de gráficos</font>

Esta celda hace que **cada gráfico que se dibuje de aquí en adelante se guarde como imagen** en la carpeta `graficos_hito3/` (matplotlib/seaborn como PNG; plotly como PNG, o como HTML interactivo si el entorno no puede exportar PNG). Al final del notebook (sección 13) se comprimen y se descargan. **Ejecutarla antes de los gráficos.**

In [ ]:
import os, re, subprocess, sys, unicodedata
import matplotlib.pyplot as plt
import plotly.io as pio
from plotly.basedatatypes import BaseFigure

CARPETA_FIG = "graficos_hito3"
os.makedirs(CARPETA_FIG, exist_ok=True)
_contador = {"n": 0}

def _nombre(titulo):
    """Nombre de archivo numerado a partir del título del gráfico."""
    _contador["n"] += 1
    sin_tildes = unicodedata.normalize("NFKD", titulo or "grafico").encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^a-z0-9]+", "_", sin_tildes.lower()).strip("_")[:60] or "grafico"
    return f"{_contador['n']:02d}_{limpio}"

# matplotlib / seaborn
def _guardar_figuras_mpl():
    for num in plt.get_fignums():
        fig = plt.figure(num)
        if getattr(fig, "_ya_guardada", False):
            continue
        titulo = fig._suptitle.get_text() if fig._suptitle is not None else ""
        if not titulo and fig.axes:
            titulo = fig.axes[0].get_title()
        fig.savefig(os.path.join(CARPETA_FIG, _nombre(titulo) + ".png"), dpi=200, bbox_inches="tight")
        fig._ya_guardada = True

_show_original = plt.show
def _show_guardando(*args, **kwargs):
    _guardar_figuras_mpl()
    return _show_original(*args, **kwargs)
plt.show = _show_guardando

# Las celdas que dibujan sin llamar a plt.show() también se guardan: se engancha al final de cada celda.
try:
    _ip = get_ipython()
    _ip.events.register("post_execute", _guardar_figuras_mpl)
    _cbs = _ip.events.callbacks["post_execute"]
    _cbs.insert(0, _cbs.pop())          # que corra antes de que el notebook cierre las figuras
except Exception as e:
    print("Aviso: no se pudo enganchar el guardado automático de figuras sin plt.show():", e)

# ---------- plotly ----------
def _puede_exportar_png():
    try:
        pio.to_image({"data": [{"type": "scatter", "x": [0], "y": [0]}]}, format="png")
        return True
    except Exception:
        return False

if not _puede_exportar_png():
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaleido==0.2.1"], check=False)
    except Exception:
        pass
PLOTLY_PNG = _puede_exportar_png()
print("Plotly se guardará como:", "PNG" if PLOTLY_PNG else "HTML interactivo (no hay exportador de PNG disponible)")

_plotly_show_original = BaseFigure.show
def _plotly_show_guardando(self, *args, **kwargs):
    titulo = self.layout.title.text if self.layout.title and self.layout.title.text else "grafico_plotly"
    base = os.path.join(CARPETA_FIG, _nombre(titulo))
    try:
        if PLOTLY_PNG:
            self.write_image(base + ".png", scale=2)
        else:
            self.write_html(base + ".html")
    except Exception as e:
        self.write_html(base + ".html")
    return _plotly_show_original(self, *args, **kwargs)
BaseFigure.show = _plotly_show_guardando

### 1.0 Carga de las 4 fuentes

In [ ]:

# Fuente 1: Padrón (puente) ---
# Codepage cp850: el .dbf viene con codepage ID=0x2 (DOS Latin-1 / cp850),
# necesario para que tildes y eñes se lean correctamente.
cols_padron = ['COD_MOD','ANEXO','NIV_MOD','D_NIV_MOD','CODGEO','DPTO','PROV','DIST',
               'D_GESTION','D_COD_TUR','CEN_EDU','DRE_UGEL','TALUMNO','AREA_CENSO','DAREACENSO']
tabla = DBF(RUTA + "Padron.dbf", encoding="cp850", load=False)
padron = pd.DataFrame([{c: rec[c] for c in cols_padron} for rec in tabla])

print("Fuente 1: padron")
display(padron.head())


In [ ]:

# Fuente 2: Matrícula y Trayectoria 2023
matricula_2023 = pd.read_csv(RUTA + "Matriculación y Trayectoria Estudiantil 2023.csv")

print("Fuente 2: matricula_2023")
display(matricula_2023.head(500))


In [ ]:

# Fuentes 3 y 4: Deserción Primaria / Secundaria (a nivel distrito, cohorte 2023-2024)
desercion_primaria = pd.read_csv(RUTA + "Tasa y número de desertores en Educación Primaria 2023-2024.csv")
desercion_secundaria = pd.read_csv(RUTA + "Tasa y número de desertores en Educación Secundaria 2023-2024.csv")

print("Fuente 3: desercion_primaria")
display(desercion_primaria.head())

print("Fuente 4: desercion_secundaria")
display(desercion_secundaria.head())


**Dimensiones de cada fuente de datos**

In [ ]:
# @title

import pandas as pd

# 1. Crear una lista con la información de cada DataFrame
datos = []
for nombre, df in [("padron", padron), ("matricula_2023", matricula_2023),
                    ("desercion_primaria", desercion_primaria),
                    ("desercion_secundaria", desercion_secundaria)]:
    # df.shape devuelve una tupla: (filas, columnas)
    datos.append({
        "Fuente": nombre,
        "Filas": df.shape[0],
        "Columnas": df.shape[1],
        "Dimensiones (Filas, Columnas)": str(df.shape)
    })

# 2. Convertir la lista en un DataFrame
df_dimensiones = pd.DataFrame(datos)

# 3. Mostrar la tabla resultante
display(df_dimensiones)

**Tipos de datos**

In [ ]:
# @title

import pandas as pd

# 1. Crear una lista para almacenar la información de las columnas y sus tipos
datos_dtypes = []

for nombre, df in [("padron", padron), ("matricula_2023", matricula_2023),
                    ("desercion_primaria", desercion_primaria)]:
    # df.dtypes devuelve una Serie donde el índice es la columna y el valor es el tipo de dato
    df_tipo = df.dtypes.reset_index()
    df_tipo.columns = ["Columna", "Tipo de Dato"] # Renombrar las columnas
    df_tipo["Fuente"] = nombre                    # Agregar una columna con el nombre de la fuente

    # Reordenar las columnas para que "Fuente" quede primero (opcional)
    df_tipo = df_tipo[["Fuente", "Columna", "Tipo de Dato"]]

    datos_dtypes.append(df_tipo)

# 2. Unir todas las tablas en una sola
df_dtypes_global = pd.concat(datos_dtypes, ignore_index=True)

# 3. Mostrar la tabla resultante
display(df_dtypes_global)

---

En la tabla **Padrón**, cada fila representa un colegio, identificado mediante la combinación de **cod_mod** y **anexo**.

En la tabla **Matrícula 2023**, cada fila también representa un colegio, utilizando **cod_mod** y **anexo** como identificadores.

Por otro lado, en las tablas de **Deserción Primaria** y **Deserción Secundaria**, cada fila representa una **ubicación geográfica**, identificada mediante el **ubigeo**, que permite determinar el departamento, provincia y distrito correspondiente.

---

# **3. Diágnostico y tratamiento de calidad**

---

### <font color=279CF5>1.1 Diagnóstico de valores faltantes por fuente</font>

No hacemos limpieza profunda aún, solo detectamos y registramos el problema.

**Cantidad de valores nulos**

In [ ]:
for nombre, df in [("padron", padron), ("matricula_2023", matricula_2023),
                    ("desercion_primaria", desercion_primaria), ("desercion_secundaria", desercion_secundaria)]:

    print(nombre)
    display(df.isna().sum().reset_index(name="Nulos"))


**Nota:** en `matricula_2023` la columna `TipoDiscaIntegrada` tiene muchos
nulos porque solo se llena cuando `Discapacidad > 0`; no es un dato faltante real, sino
estructuralmente ausente (categoría "sin discapacidad registrada").


### <font color=279CF5>1.2 Valores distintos en variables categóricas clave</font>

**Análisis de variables categóricas**

In [ ]:

print("Niveles educativos en matrícula (dsc_nivel):")
display(matricula_2023["dsc_nivel"].value_counts(dropna=False))

print("\nTipo de gestión en matrícula:")
display(matricula_2023["gestion"].value_counts(dropna=False))

print("\nNiveles modulares en el padrón (D_NIV_MOD):")
display(padron["D_NIV_MOD"].value_counts(dropna=False))

print("\nGestión en el padrón (D_GESTION):")
display(padron["D_GESTION"].value_counts(dropna=False))


**Valores duplicados**

In [ ]:
import pandas as pd

for nombre, df in [("padron", padron), ("matricula_2023", matricula_2023),
                    ("desercion_primaria", desercion_primaria),
                    ("desercion_secundaria", desercion_secundaria)]:

    # Calcular duplicados por columna para la fuente actual
    datos_col = []
    for col in df.columns:
        n_dup_col = df[col].duplicated().sum()
        datos_col.append({
            "Columna": col,
            "Valores Duplicados": n_dup_col,
            "Total Filas": len(df)
        })

    # Crear el DataFrame de la fuente
    df_resultado = pd.DataFrame(datos_col)

    # Mostrar título y la tabla correspondiente
    print(f" Duplicados por columna en: {nombre} ")
    display(df_resultado)
    print("\n")

**Duplicados exactos (fila completa) en cada fila**

In [ ]:
import pandas as pd

# 1. Recopilar los duplicados exactos por fuente
datos_resumen = []

for nombre, df in [("padron", padron), ("matricula_2023", matricula_2023),
                    ("desercion_primaria", desercion_primaria),
                    ("desercion_secundaria", desercion_secundaria)]:

    n_dup = df.duplicated().sum()

    datos_resumen.append({
        "Fuente": nombre,
        "Filas Duplicadas Exactas": n_dup,
        "Total de Filas": len(df)
    })

#  Convertir a DataFrame global
df_duplicados_exactos = pd.DataFrame(datos_resumen)

# Mostrar la tabla con formato claro y separador de miles
display(
    df_duplicados_exactos.style
    .format({"Filas Duplicadas Exactas": "{:,}", "Total de Filas": "{:,}"})
    .hide(axis="index")
)


# <font color=green>Parte 2. Organizar datos antes de integrar</font>

Necesitamos:
1. Filtrar la matrícula a **Primaria** y **Secundaria** (para que sea comparable con los archivos
   de deserción, que solo cubren esos dos niveles).
2. Normalizar las llaves de unión (`cod_mod`, `anexo`, `ubigeo`) para que tengan el mismo formato
   en todas las fuentes.


**2.1 Filtrar matrícula a Primaria y Secundaria**

In [ ]:
import pandas as pd

# 1. Filtrar solo Primaria y Secundaria
matricula = matricula_2023[matricula_2023["dsc_nivel"].isin(["Primaria", "Secundaria"])].copy()

# 2. Crear una tabla simple para comparar tamaños
df_resultado = pd.DataFrame({
    "Estado": ["Original", "Filtrada"],
    "Total Filas": [len(matricula_2023), len(matricula)]
})

# 3. Mostrar la tabla
display(df_resultado)

**2.2 Adecuación de las variables llaves para el merge (Patrón y matricula)**

Código del colegio (cod_mod): Convierte el código modular a texto y completa con ceros a la izquierda hasta obtener 7 dígitos (.str.zfill(7)), asegurando que códigos como 0123456 y 123456 puedan coincidir correctamente.

Anexo (anexo): Convierte los valores del anexo a texto en ambas tablas, permitiendo que tengan el mismo formato al momento de realizar el cruce de datos.

Código de ubicación geográfica (ubigeo / CODGEO): Convierte los códigos a texto y completa con ceros a la izquierda hasta obtener 6 dígitos (.str.zfill(6)), evitando la pérdida de ceros iniciales al leerlos como números.

In [ ]:

#  - cod_mod en matrícula es numérico (int64); en el padrón es texto con cero a la izquierda (7 dígitos)
#  - anexo en ambas fuentes es numérico/entero, pero en el padrón viene como texto
#  - CODGEO / ubigeo: 6 dígitos con cero a la izquierda

matricula["cod_mod_key"] = matricula["cod_mod"].astype(str).str.zfill(7)
matricula["anexo_key"] = matricula["anexo"].astype(str)

padron["cod_mod_key"] = padron["COD_MOD"].astype(str).str.zfill(7)
padron["anexo_key"] = padron["ANEXO"].astype(str)

# ubigeo de deserción viene sin cero a la izquierda al leerse como int
desercion_primaria["ubigeo_key"] = desercion_primaria["ubigeo"].astype(str).str.zfill(6)
desercion_secundaria["ubigeo_key"] = desercion_secundaria["ubigeo"].astype(str).str.zfill(6)
padron["ubigeo_key"] = padron["CODGEO"].astype(str).str.zfill(6)

matricula[["cod_mod","cod_mod_key","anexo","anexo_key"]].head()


### <font color=279CF5>2.3 Tabla resumen de cada fuente</font>
Un resumen rápido, previo a integrar, ayuda a detectar problemas de escala o de cobertura.

In [ ]:
print("Resumen 1 - Padrón: escuelas por nivel modular")
display(padron.groupby("D_NIV_MOD").size().reset_index(name="n_escuelas").sort_values("n_escuelas", ascending=False))


In [ ]:
print("Resumen 2 - Matrícula 2023: total de estudiantes matriculados por nivel")
resumen_matricula = matricula.groupby("dsc_nivel")["TotalEstudiantes"].sum().reset_index()
display(resumen_matricula)


In [ ]:
print("Resumen 3 - Matrícula 2023: estudiantes por tipo de gestión y nivel")
resumen_gestion = matricula.groupby(["gestion","dsc_nivel"])["TotalEstudiantes"].sum().reset_index()
display(resumen_gestion)


In [ ]:
print("Resumen 4 - Movilización de estudiantes de Primaria: desertores y tasa promedio por departamento")
resumen_des_prim = desercion_primaria.groupby("Departamento").agg(
    total_estudiantes_movilizados=("desertor","sum"),
    total_matriculados=("denominador","sum"),
    tasa_promedio=("Tasa","mean")
).round(2).sort_values("total_estudiantes_movilizados", ascending=False)
display(resumen_des_prim.head(10))


In [ ]:
print("Resumen 5 - Deserción Secundaria: desertores y tasa promedio por departamento")
resumen_des_sec = desercion_secundaria.groupby("Departamento").agg(
    total_desertores=("desertor","sum"),
    total_matriculados=("denominador","sum"),
    tasa_promedio=("Tasa","mean")
).round(2).sort_values("total_desertores", ascending=False)
display(resumen_des_sec.head(10))



# <font color=green>Parte 3. `concat` versus `merge`</font>

- Usamos **`concat`** más adelante para apilar `desercion_primaria` y `desercion_secundaria`,
  que comparten exactamente las mismas columnas (Parte 7).
- Usamos **`merge`** para emparejar fuentes con información complementaria sobre una misma unidad
  de análisis usando una llave común: es el caso de matrícula ↔ padrón, y de la base agregada
  ↔ deserción.



# <font color=green>Parte 4. Definir la unidad de análisis y las llaves</font>

**Nivel distrito x nivel educativo**: agregamos la base anterior por distrito y la
   emparejamos con los archivos de deserción, cuya unidad de análisis ya es el distrito
   (`ubigeo`) para un nivel educativo dado.



# <font color=green>Parte 5. Integración auditada (Nivel 1): Matrícula ↔ Padrón</font>

Claves de unión: `cod_mod_key`, `anexo_key`, `id_nivel` (en matrícula) y `NIV_MOD` (en padrón)´

Cardinalidad: `matricula`: Múltiples filas por escuela + nivel + edad. `padron`: Una fila por escuela + nivel.

Tipo de Unión: `how="left"` porque queremos conservar todas las filas de matrícula y solo agregar
la geografía cuando exista coincidencia en el padrón; con `indicator=True` podemos auditar
cuántas filas cruzaron.

Granularidad resultante: `matricula_geo` mantiene la granularidad de `matricula` (escuela x nivel x edad)


In [ ]:
matricula_geo = matricula.merge(
    padron[["cod_mod_key","anexo_key","NIV_MOD","D_NIV_MOD","CODGEO","ubigeo_key",
             "DPTO","PROV","DIST","D_GESTION","DRE_UGEL","AREA_CENSO","DAREACENSO"]],
    left_on=["cod_mod_key","anexo_key","id_nivel"],
    right_on=["cod_mod_key","anexo_key","NIV_MOD"],
    how="left",
    indicator=True
)

conteo_cruce_1 = matricula_geo["_merge"].value_counts().reset_index()
conteo_cruce_1.columns = ["resultado_cruce","cantidad"]
display(conteo_cruce_1)


In [ ]:
fig1 = px.bar(
    conteo_cruce_1,
    x="resultado_cruce",
    y="cantidad",
    text="cantidad",
    title="Nivel 1 - Cruce Matrícula ↔ Padrón (llave: escuela + nivel)"
)
fig1.show()


**Interpretación:**

**both (311,015 filas, 79.12%):** filas de matrícula que encontraron su escuela y nivel en el padrón, por lo que se les pudo asignar departamento, provincia, distrito y ubigeo.

**left_only (82,090 filas, 20.88%):** filas de matrícula cuya combinación `cod_mod + anexo + nivel` no apareció en el padrón. No se les pudo asignar ubicación geográfica.

**right_only (0 filas)** es esperado, porque con `how="left"` no se conservan filas exclusivas del padrón.


**Precaución:** las filas sin geografía (`left_only`) no se pueden ubicar en un distrito y por
lo tanto **no podrán cruzar** con los archivos de deserción en el siguiente paso. Se conservan
en `matricula_geo` para no perder información de matrícula, pero se excluirán al construir la
base agregada por distrito.



# <font color=green>Parte 6. Agregar matrícula 2023 a nivel distrito x nivel</font>

La deserción se define como *matriculado en 2023 que no volvió a matricularse en 2024*, y su
`denominador` es esa matrícula 2023. Para poder cruzarla, agregamos la matrícula (ya
georreferenciada) a nivel distrito x nivel educativo.


La matrícula tiene varias filas por distrito, ya que registra cada escuela y edad, mientras que la deserción tiene una sola fila por distrito y nivel. Por eso, primero se agrupa la matrícula para obtener un solo registro por distrito y nivel. Así, el merge se realizará correctamente y se pueden comparar ambas bases sin repetir los datos de deserción.

In [ ]:

base_distrital = (
    matricula_geo[matricula_geo["_merge"] == "both"]
    .groupby(["ubigeo_key","DPTO","PROV","DIST","dsc_nivel"], as_index=False)
    .agg(
        total_estudiantes_2023=("TotalEstudiantes","sum"),
        total_mujeres_2023=("Mujer","sum"),
        total_hombres_2023=("Hombre","sum"),
        total_aprobados_2023=("Aprobado","sum"),
        total_desaprobados_2023=("Desaprobado","sum"),
        total_retirados_2023=("Retirado","sum"),
        n_escuelas_2023=("cod_mod_key","nunique"),
        total_discapacidad_2023 =("Discapacidad","sum")
    )
)

print("Base agregada a nivel distrito x nivel (matrícula 2023):", base_distrital.shape)

# **4. Integración**

---



# <font color=green>Parte 7. Preparar e integrar los archivos de deserción</font>

Ambos archivos de deserción comparten estructura (`ubigeo`, `Departamento`, `Provincia`,
`Distrito`, `desertor`, `denominador`, `Tasa`), pero cada uno corresponde a **un nivel
educativo distinto** (Primaria / Secundaria). Recordemos: `denominador` = matriculados **2023**
y `desertor` = de ellos, quienes no se volvieron a matricular en **2024**. Apilamos ambos
archivos con `concat`, agregando una columna `dsc_nivel`, para luego cruzarlos con
`base_distrital` (que ya está construida solo con matrícula 2023) usando la llave
`ubigeo + nivel`.


Se usa concat porque los archivos de deserción de Primaria y Secundaria tienen las mismas columnas y representan el mismo tipo de información.

concat permite unirlos verticalmente, es decir, colocar las filas de Secundaria debajo de las filas de Primaria.

In [ ]:

desercion_primaria["dsc_nivel"] = "Primaria"
desercion_secundaria["dsc_nivel"] = "Secundaria"

desercion = pd.concat([desercion_primaria, desercion_secundaria], ignore_index=True)
desercion = desercion.rename(columns={
    "desertor": "n_desertores",
    "denominador": "n_matriculados_base_desercion",
    "Tasa": "tasa_desercion_pct"
})

print("Deserción (Primaria + Secundaria) concatenada:", desercion.shape)
desercion.head()


Los archivos de deserción no cubren exactamente los mismos distritos (1,890 en Primaria y 1,846 en Secundaria), por lo que algunos distritos solo cuentan con información de un nivel. No se imputaron los valores faltantes, ya que en estos casos la ausencia probablemente refleja que el distrito no ofrece ese nivel, y rellenarla supondría inventar información.

Se valida que ubigeo + dsc_nivel no esté duplicado en la base de deserción.

In [ ]:

# Validar duplicados en la llave ubigeo + nivel
dup_desercion = desercion.duplicated(subset=["ubigeo_key","dsc_nivel"]).sum()
print("Duplicados en deserción (ubigeo+nivel):", dup_desercion)


### <font color=279CF5>7.1 Integración auditada (Nivel 2): Base distrital ↔ Deserción</font>

Claves de unión: `ubigeo_key` y `dsc_nivel`

Cardinalidad: `base_distrital`: Una fila por distrito + nivel. `desercion`: Una fila por distrito + nivel.

Tipo de Unión: how="outer" para validación, luego filtrado a both para `base_final`

Granularidad resultante: `base_distrital` (antes del merge): Distrito x Nivel Educativo. `base_final`: Distrito x Nivel Educativo

In [ ]:
base_validacion_2 = base_distrital.merge(
    desercion[["ubigeo_key","dsc_nivel","n_desertores","n_matriculados_base_desercion","tasa_desercion_pct"]],
    on=["ubigeo_key","dsc_nivel"],
    how="outer",
    indicator=True,
    suffixes=("_matricula","_desercion")
)

conteo_cruce_2 = base_validacion_2["_merge"].value_counts().reset_index()
conteo_cruce_2.columns = ["resultado_cruce","cantidad"]
conteo_cruce_2["porcentaje"] = 100 * conteo_cruce_2["cantidad"] / conteo_cruce_2["cantidad"].sum()
display(conteo_cruce_2)

In [ ]:

fig2 = px.bar(
    conteo_cruce_2,
    x="resultado_cruce",
    y="cantidad",
    text="cantidad",
    title="Nivel 2 - Cruce Base distrital (Matrícula+Padrón) ↔ Deserción (llave: ubigeo + nivel)"
)
fig2.show()

In [ ]:
print("Registros que no cruzan entre la base distrital y deserción:")
no_cruzan_2 = base_validacion_2[base_validacion_2["_merge"] != "both"]
display(no_cruzan_2[["ubigeo_key","DPTO","DIST","dsc_nivel","_merge"]].head(15))


 `left_only` son distritos con matrícula pero sin dato de deserción reportado
(posiblemente por secreto estadístico en distritos muy pequeños); `right_only` son distritos
con deserción reportada pero cuya matrícula no logró georreferenciarse en el Nivel 1 (por
ejemplo, escuelas sin coincidencia exacta en el padrón). Ambos casos deben excluirse del
análisis de asociación matrícula-deserción, pero se documentan aquí en vez de descartarse en
silencio.



# <font color=green>Parte 8. Construir la base integrada final para análisis</font>

Nos quedamos con los distritos x nivel que **sí cruzan** en ambos niveles de integración
(`_merge == "both"`), que es la base con la que podemos estudiar la relación entre indicadores
de matrícula/trayectoria y la deserción escolar.


In [ ]:

base_final = base_validacion_2[base_validacion_2["_merge"] == "both"].drop(columns="_merge").copy()
base_final["pct_discapacidad"] = base_final["total_discapacidad_2023"] / base_final["total_estudiantes_2023"]
print("Tamaño de la base integrada final:", base_final.shape)
base_final.head()


In [ ]:
# 8.1b Agregar desglose por tipo de gestión (pública / privada) a la base final
# La matrícula original trae la columna "gestion". La agregamos a nivel
# distrito x nivel x gestión y la pasamos a formato ancho (una columna por
# tipo de gestión) para no alterar la granularidad de base_final
# (distrito x nivel), que es la que se usó para cruzar con deserción.

gestion_distrital = (
    matricula_geo[matricula_geo["_merge"] == "both"]
    .groupby(["ubigeo_key", "dsc_nivel", "gestion"], as_index=False)
    .agg(total_estudiantes=("TotalEstudiantes", "sum"))
)

gestion_wide = gestion_distrital.pivot_table(
    index=["ubigeo_key", "dsc_nivel"],
    columns="gestion",
    values="total_estudiantes",
    fill_value=0
).reset_index()

# Prefijo para identificar que son columnas de estudiantes por gestión
gestion_wide.columns = [
    col if col in ["ubigeo_key", "dsc_nivel"] else f"estudiantes_2023_{col}"
    for col in gestion_wide.columns
]

base_final = base_final.merge(gestion_wide, on=["ubigeo_key", "dsc_nivel"], how="left")

print("Base final con desglose por gestión:", base_final.shape)
base_final.head()

**Diccionario**

In [ ]:
diccionario = {
    "ubigeo_key": ("Código UBIGEO de 6 dígitos del distrito.", "Cualitativa nominal"),
    "DPTO": ("Departamento del Perú.", "Cualitativa nominal"),
    "PROV": ("Provincia del Perú.", "Cualitativa nominal"),
    "DIST": ("Distrito del Perú.", "Cualitativa nominal"),
    "dsc_nivel": ("Nivel educativo: Primaria o Secundaria.", "Cualitativa nominal"),
    "total_estudiantes_2023": ("Total de estudiantes matriculados en 2023.", "Cuantitativa discreta"),
    "total_mujeres_2023": ("Total de mujeres matriculadas en 2023.", "Cuantitativa discreta"),
    "total_hombres_2023": ("Total de hombres matriculados en 2023.", "Cuantitativa discreta"),
    "total_aprobados_2023": ("Total de estudiantes aprobados en 2023.", "Cuantitativa discreta"),
    "total_desaprobados_2023": ("Total de estudiantes desaprobados en 2023.", "Cuantitativa discreta"),
    "total_retirados_2023": ("Total de estudiantes retirados en 2023.", "Cuantitativa discreta"),
    "n_escuelas_2023": ("Número de escuelas con matrícula en 2023.", "Cuantitativa discreta"),
    "total_discapacidad_2023": ("Total de estudiantes con discapacidad en 2023.", "Cuantitativa discreta"),
    "n_desertores": ("Estudiantes matriculados en 2023 que no se matricularon en 2024.", "Cuantitativa discreta"),
    "n_matriculados_base_desercion": ("Matriculados en 2023 usados como base para la tasa de deserción.", "Cuantitativa discreta"),
    "tasa_desercion_pct": ("Tasa de deserción en porcentaje (n_desertores / denominador x 100).", "Cuantitativa continua"),
    "pct_discapacidad": ("Proporción de estudiantes con discapacidad (0 a 1).", "Cuantitativa continua"),
    "estudiantes_2023_Privada": ("Estudiantes matriculados en colegios de gestión privada.", "Cuantitativa discreta"),
    "estudiantes_2023_Pública de gestión directa": ("Estudiantes matriculados en colegios públicos de gestión directa.", "Cuantitativa discreta"),
    "estudiantes_2023_Pública de gestión privada": ("Estudiantes matriculados en colegios públicos de gestión privada.", "Cuantitativa discreta"),
    "nivel_bin": ("Codificación de dsc_nivel: 0 = Primaria, 1 = Secundaria.", "Cualitativa binaria"),
    "pct_aprobados": ("Proporción de aprobados sobre el total de estudiantes (0 a 1).", "Cuantitativa continua"),
    "pct_desaprobados": ("Proporción de desaprobados sobre el total de estudiantes (0 a 1).", "Cuantitativa continua"),
    "pct_retirados": ("Proporción de retirados sobre el total de estudiantes (0 a 1).", "Cuantitativa continua"),
    "estudiantes_por_escuela": ("Promedio de estudiantes por escuela (total_estudiantes / n_escuelas).", "Cuantitativa continua"),
}

# Verifica que no falte ninguna columna
faltan = set(base_final.columns) - set(diccionario)
print("Columnas sin describir:", faltan)

diccionario_base_final = pd.DataFrame({
    "Nombre_Variable": list(base_final.columns),
    "Descripcion": [diccionario.get(c, ("(sin descripción)", ""))[0] for c in base_final.columns],
    "Tipo_Variable": [diccionario.get(c, ("", "(sin tipo)"))[1] for c in base_final.columns],
    "Tipo_Dato": [str(base_final[c].dtype) for c in base_final.columns],
})

display(diccionario_base_final)


### <font color=279CF5>8.1 Validación cruzada: ¿el denominador de deserción coincide con nuestra propia matrícula 2023?</font>

Como `n_matriculados_base_desercion` (el `denominador` del archivo de deserción) debería ser,
en teoría, la misma matrícula 2023 que calculamos de forma independiente en la Parte 6
(`total_estudiantes_2023`), comparar ambas columnas es una buena prueba de consistencia entre
fuentes: si difieren mucho, hay un problema de cobertura o de definición que hay que investigar
antes de confiar en la tasa de deserción a nivel distrito.


In [ ]:

# Cálculo puntual para esta validación (no se agrega a base_final: es solo diagnóstico)
diferencia_pct = (
    100 * (base_final["n_matriculados_base_desercion"] - base_final["total_estudiantes_2023"])
    / base_final["n_matriculados_base_desercion"]
)

print("Estadísticos de la diferencia (denominador oficial - matrícula 2023 calculada):")
display(diferencia_pct.describe().round(2))

print("\nDistritos x nivel con mayor discrepancia (top 10 en valor absoluto):")
tabla_diferencia = base_final[["DPTO","DIST","dsc_nivel","total_estudiantes_2023","n_matriculados_base_desercion"]].copy()
tabla_diferencia["diferencia_pct"] = diferencia_pct
display(tabla_diferencia.reindex(diferencia_pct.abs().sort_values(ascending=False).index).head(10))


Se comprueba que ambas columnas representan el mismo concepto: estudiantes matriculados en 2023 por distrito y nivel. Una proviene del dato oficial de deserción y la otra la calculamos a partir de la matrícula. Al compararlas y obtener una diferencia mediana de 0%, confirmamos que nuestra integración es consistente.

La integración tiene una tasa de cruce del 99.7% en ambos merges, cero duplicados y cero nulos en la base final. Esto no significa que no haya errores — documentamos 0.3% de registros que no cruzaron, y algunos distritos rurales (como Kimbiri y Pichari) muestran discrepancias notables al validar contra el denominador oficial de deserción — pero para la gran mayoría del país, la base integrada representa fielmente la matrícula y deserción reales.


# <font color=green>Parte 9. Valores nulos y outliers en la base integrada</font>


In [ ]:

print("Valores nulos en la base final:")
display(base_final.isna().sum())


In [ ]:

# Outliers con la regla de rango intercuartílico (IQR)
def resumen_outliers_iqr(serie, nombre):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_outliers = ((serie < lim_inf) | (serie > lim_sup)).sum()
    return {
        "variable": nombre, "Q1": round(q1,2), "Q3": round(q3,2),
        "limite_inferior": round(lim_inf,2), "limite_superior": round(lim_sup,2),
        "n_outliers": n_outliers, "pct_outliers": round(100*n_outliers/len(serie),2)
    }

variables_numericas = ["total_estudiantes_2023","n_desertores","tasa_desercion_pct","n_escuelas_2023"]
resumen_out = pd.DataFrame([resumen_outliers_iqr(base_final[v], v) for v in variables_numericas])
display(resumen_out)



**Nota sobre outliers:** distritos con `total_estudiantes_2023` o `n_desertores` muy altos suelen
corresponder a distritos urbanos grandes (p. ej. capitales de provincia), no a errores de
datos; antes de eliminarlos hay que revisar si el outlier es un valor genuino (heterogeneidad
real entre distritos) o un problema de calidad de datos.


# **5. EDA e interpretación**

---

# <font color=green>Parte 11. Visualizaciones</font>

### <font color=279CF5>11.1 Barras: total de estudiantes matriculados por gestión y nivel (2023)</font>

In [ ]:

fig4 = px.bar(
    resumen_gestion,
    x="gestion",
    y="TotalEstudiantes",
    color="dsc_nivel",
    barmode="group",
    text="TotalEstudiantes",
    title="Estudiantes matriculados por tipo de gestión y nivel (2023)"
)
fig4.show()


**Interpretación:** En 2023, la mayor cantidad de estudiantes matriculados se concentra en las instituciones públicas de gestión directa, con 262 5176 estudiantes en Primaria y 2025101 en Secundaria. En la gestión privada, se registran 976573 matriculados en Primaria y 691256 en Secundaria. Por otro lado, las instituciones públicas de gestión privada presentan 120103 estudiantes en Primaria y 126474 en Secundaria. En general, la gestión pública directa concentra la mayor cantidad de estudiantes matriculados en ambos niveles educativos.

### <font color=279CF5>11.2 Top 10 Distritos con mayor tasa de movilidad/salida escolar(2023)</font>

In [ ]:
distrito = (base_final.groupby(["ubigeo_key", "DPTO", "DIST"], as_index=False)
            .agg(n_desertores=("n_desertores", "sum"),
                 denominador=("n_matriculados_base_desercion", "sum")))
distrito["tasa_general"] = 100 * distrito["n_desertores"] / distrito["denominador"]

# Filtro para evitar distritos diminutos (tasas inestables)
distrito = distrito[distrito["denominador"] >= 60]

top = distrito.nlargest(10, "tasa_general")

fig = px.bar(top, x="tasa_general", y="DIST", color="DPTO", orientation="h",
             text="tasa_general",
             title="Top 10 distritos con mayor tasa de movilidad/salida escolar (Primaria y Secundaria, 2023-2024)",
             labels={"tasa_general": "Tasa de salida (%)", "DIST": "Distrito", "DPTO": "Departamento"})
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

**Interpretación:** El gráfico muestra los diez distritos con mayor tasa de mmovilización de estudiantes de primaria y secundaria. San Cristóbal (Moquegua) con 14.81%, seguido por Teniente Manuel Clavero
(Loreto) con 12.81% y Santa María del Mar (Lima) con 10.59%.

# **6. Transformación de variables**

---

**6.1 Codificación**

Codificamos dsc_nivel como variable binaria (nivel_bin), asignando 0 a Primaria y 1 a Secundaria, dado que al tener únicamente dos categorías no es necesario aplicar one-hot encoding.

In [ ]:
#Codificación binaria de dsc_nivel (Primaria/Secundaria). Como es una variable categorica ordinal
#como solo Primaria o Secundaria la consideramos como binaria
base_final["nivel_bin"] = base_final["dsc_nivel"].map({"Primaria": 0, "Secundaria": 1})

**6.2 Variables derivadas**

Creamos cuatro variables derivadas a partir de los totales de la matrícula 2023: el porcentaje de aprobados (pct_aprobados), el porcentaje de desaprobados (pct_desaprobados), el porcentaje de retirados (pct_retirados) y el número de estudiantes por escuela (estudiantes_por_escuela), este último como indicador del tamaño promedio de las instituciones educativas en cada distrito.

In [ ]:
base_final["pct_desaprobados"] = base_final["total_desaprobados_2023"] / base_final["total_estudiantes_2023"]
base_final["pct_retirados"] = base_final["total_retirados_2023"] / base_final["total_estudiantes_2023"]
base_final["estudiantes_por_escuela"] = base_final["total_estudiantes_2023"] / base_final["n_escuelas_2023"]


In [ ]:
base_final

# **7. Matriz analítica**

---

**7.1 Variables incluidas, representación final y exclusiones importantes**
La unidad de análisis es **distrito × nivel educativo**: cada fila representa un
distrito en un nivel (primaria o secundaria), identificada por `ubigeo_key` +
`nivel_bin`. El nivel se codificó como variable binaria (0 = Primaria, 1 = Secundaria).

Variables incluidas: identificadores (`ubigeo_key`, `DPTO`, `PROV`, `DIST`), `nivel_bin`,
`pct_desaprobados`, `pct_retirados`, `pct_discapacidad`, `tasa_desercion_pct` y
`estudiantes_por_escuela`. Se excluye `pct_aprobados` porque, junto con desaprobados y
retirados, suma aproximadamente 1 (colinealidad). Se excluyen los totales absolutos y el
desglose por gestión, porque el análisis compara tasas y proporciones, no volúmenes.

In [ ]:
matriz_analitica = base_final[["ubigeo_key","DPTO","PROV","DIST","nivel_bin",
                               "pct_desaprobados","pct_retirados","pct_discapacidad",
                               "tasa_desercion_pct","estudiantes_por_escuela"]].copy()

In [ ]:
matriz_analitica

**7.2 Discretización o escalamiento solo cuando correspondan y estén justificados**

**Escalamiento.** Se aplicó z-score a `pct_desaprobados`, `pct_retirados`,
`tasa_desercion_pct` y `estudiantes_por_escuela`. Sin escalar, esta última dominaría el cálculo de las distancias. El
z-score deja cada variable con media 0 y desviación 1, de modo que todas pesan de forma comparable.

**Transformación logarítmica (solo en tamaño de escuela).** `estudiantes_por_escuela` y `pct_discapacidad`
tienen una distribución muy asimétrica: pocos distritos urbanos con escuelas muy grandes, y pocos alumnos con dispacidad.
y muchos distritos rurales con escuelas pequeñas. Se aplicó `log1p` antes del z-score
para reducir ese sesgo.

**Qué no se escala**
- `nivel_bin`, porque ya es binaria (0 = Primaria, 1 = Secundaria) y no necesita
  transformación.
- Los identificadores (`ubigeo_key`, `DPTO`, `PROV`, `DIST`), porque no son variables
  numéricas de análisis.

**Discretización (no se aplica).** Convertir las tasas en categorías como "alta, media,
baja" no es necesario.

**Valores nulos.** La matriz no tiene nulos, por lo que no se requirió imputación ni
eliminación de filas antes del escalamiento.

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Variables a escalar (excluimos nivel_bin porque ya es binaria 0/1,
# y ubigeo_key, DPTO, PROV, DIST porque son identificadores)
variables_escalar = ["pct_desaprobados", "pct_retirados", "pct_discapacidad",
                     "tasa_desercion_pct", "estudiantes_por_escuela"]

# --- 1. Matriz original sin escalar (referencia) ---
matriz_analitica_final = matriz_analitica.copy()

# --- 2. Matriz con Z score (log1p al tamaño por su asimetría) ---
matriz_final_zscore = matriz_analitica.copy()
matriz_final_zscore["estudiantes_por_escuela"] = np.log1p(matriz_final_zscore["estudiantes_por_escuela"])
matriz_final_zscore["pct_discapacidad"] = np.log1p(matriz_final_zscore["pct_discapacidad"])  # puede ser asimetrica

estandarizador = StandardScaler()
matriz_final_zscore[variables_escalar] = (
    estandarizador.fit_transform(
        matriz_final_zscore[variables_escalar]
    )
)

# --- Comparamos las dos representaciones ---
print("Matriz sin escalar:", matriz_analitica_final.shape)
display(matriz_analitica_final.head())

print("\nMatriz con Z-score:", matriz_final_zscore.shape)
display(matriz_final_zscore.head())

# **8. Reducción de dimensionalidad**

---

**8.1 Seleccionar atributos**

In [ ]:
atributos = ["pct_desaprobados", "pct_retirados", "pct_discapacidad",
             "tasa_desercion_pct", "estudiantes_por_escuela"]

# Misma transformación log1p que en 7.2
X = matriz_analitica[atributos].copy()
X[["estudiantes_por_escuela", "pct_discapacidad"]] = np.log1p(
    X[["estudiantes_por_escuela", "pct_discapacidad"]])

nivel = matriz_analitica["nivel_bin"].map({0: "Primaria", 1: "Secundaria"})

print("Dimensiones de X:", X.shape)
print("Nulos:", X.isna().sum().sum())
display(X.describe().T.round(2))

**8.2 Aplicar PCA**

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns
# 1. Estandarización
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X)
print("Media aprox.:", np.round(X_std.mean(axis=0), 2))
print("Desv. estándar aprox.:", np.round(X_std.std(axis=0), 2))

# 2. Componentes principales (todas)
pca = PCA()
Z = pca.fit_transform(X_std)
P = pca.components_.T          # p x k: cada columna es una dirección de proyección

# 3. Varianza explicada y acumulada
nombres_z = [f"Z{i+1}" for i in range(len(atributos))]
tabla_varianza = pd.DataFrame({"Componente": nombres_z,
                               "Proporcion": pca.explained_variance_ratio_})
tabla_varianza["Porcentaje"] = 100 * tabla_varianza["Proporcion"]
tabla_varianza["Acumulada"] = 100 * tabla_varianza["Proporcion"].cumsum()
display(tabla_varianza.round(2))

n80 = int(np.argmax(tabla_varianza["Acumulada"] >= 80) + 1)
print(f"Componentes necesarias para conservar ≥80% de la varianza: {n80}")

from matplotlib import pyplot as plt
plt.figure(figsize=(10,10))

ax = sns.barplot(data=tabla_varianza, x="Componente", y="Porcentaje", hue="Componente", legend=False)
# Etiquetas con el porcentaje sobre cada barra
for contenedor in ax.containers:
    ax.bar_label(contenedor, fmt="%.2f%%", padding=3)

ax.set(title="Varianza explicada por componente", xlabel="", ylabel="Varianza explicada (%)")
ax.figure.set_size_inches(7, 3)

El primer componente explica el 37.00% de la varianza y los dos primeros acumulan 58.88%. Se requieren 4 componentes para superar el 80% (91.11%). La varianza decrece de forma gradual, sin una caída marcada entre componentes, lo que indica que las cinco variables aportan información en gran parte distinta (están poco correlacionadas). Por ello, el PCA no reduce mucho la dimensionalidad.

In [ ]:
X.corr().round(2)

In [ ]:
# 4. Direcciones de proyección de Z1 y Z2
P_df = pd.DataFrame(P, index=atributos, columns=nombres_z)
display(P_df[["Z1", "Z2"]].round(2))

ax = sns.heatmap(P_df[["Z1", "Z2"]], annot=True, fmt=".2f", cmap="vlag", center=0)
ax.set(title="Direcciones de proyección de PCA", xlabel="", ylabel="")
ax.figure.set_size_inches(6, 4)

**Interpretación de Z1 y Z2.**

Z1 explica el 37.00% de la varianza y tiene pesos positivos y
similares en `tasa_desercion_pct` (0.61), `pct_retirados` (0.53), `pct_desaprobados` (0.43) y
`estudiantes_por_escuela` (0.40), mientras que `pct_discapacidad` casi no participa (−0.01).
Por tanto, Z1 representa un índice general de salida escolar y trayectoria débil: valores
altos corresponden a distritos × nivel con mayor deserción, retiros y desaprobación, y con
escuelas más grandes.

Z2 explica el 21.88% de la varianza y está dominada por `pct_discapacidad` (0.83), en
contraste con `estudiantes_por_escuela` (−0.43), de modo que distingue los casos con alta
proporción de estudiantes con discapacidad y escuelas pequeñas de los de escuelas grandes.

In [ ]:
# 5. Representación Z1-Z2 (nivel solo para colorear)
Z_df = pd.DataFrame(Z[:, :2], columns=["Z1", "Z2"], index=matriz_analitica.index)
Z_df["Nivel"] = nivel

ax = sns.scatterplot(data=Z_df, x="Z1", y="Z2", hue="Nivel", s=30, alpha=0.6)
ax.set_title("PCA de los distritos × nivel")
ax.figure.set_size_inches(7, 5)

**Representación PCA.** En el plano Z1-Z2 los distritos × nivel forman una nube continua,
densa en la zona de valores bajos y con una cola hacia valores altos. Aunque `nivel_bin` no
se utilizó para construir las componentes, la primaria se concentra en los valores bajos de
Z1 y la secundaria se extiende hacia los altos, lo que indica que Z1 (salida escolar, retiros,
desaprobación y tamaño de escuela) es en promedio mayor en secundaria. Existe una zona
amplia de superposición, por lo que el nivel no separa los casos por completo. Z2 no
diferencia los niveles. No se observan grupos claramente separados, y se aprecian algunos
valores extremos en ambas componentes.

**8.3 Aplicar NMF sobre los mismos atributos**



In [ ]:
# 1. Min-Max (NMF exige valores no negativos)
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import NMF

scaler_mm = MinMaxScaler()
X_pos = scaler_mm.fit_transform(X)
print("Mínimo:", X_pos.min(), "| Máximo:", X_pos.max())

# 2. NMF con 2 factores
nmf = NMF(n_components=2, init="random", random_state=1, max_iter=2000)
W = nmf.fit_transform(X_pos)
H = nmf.components_
print("X_pos:", X_pos.shape, "| W:", W.shape, "| H:", H.shape)
print("Error de reconstrucción NMF:", round(nmf.reconstruction_err_, 3))

In [ ]:
# 3. Matriz H: composición de los factores
H_df = pd.DataFrame(H, index=["NMF1", "NMF2"], columns=atributos)
display(H_df.round(2))

ax = sns.heatmap(H_df, annot=True, fmt=".2f", cmap="Blues")
ax.set(title="Matriz H: composición de los factores NMF", xlabel="Atributos originales", ylabel="Factor")
ax.figure.set_size_inches(9, 3)

**Interpretación de H.** El factor NMF1 está formado casi exclusivamente por
`estudiantes_por_escuela` (1.19); el resto de atributos tiene pesos cercanos a 0
(`pct_discapacidad` 0.12, `pct_desaprobados` 0.06, `pct_retirados` y `tasa_desercion_pct`
0.00). Representa un perfil de tamaño de las escuelas. El factor NMF2 está dominado por
`tasa_desercion_pct` (2.25), `pct_desaprobados` (1.69) y `pct_retirados` (0.88), con una
participación menor de `estudiantes_por_escuela` (0.74) y `pct_discapacidad` (0.43). Representa
la movilidad de escolares. Así, NMF separa dos dimensiones:
el tamaño de la escuela y el riesgo de movilidad.

**Interpretación de los factores NMF (matriz H).**

| Atributo | NMF1 | NMF2 |
|---|---|---|
| pct_desaprobados | 0.06 | **1.69** |
| pct_retirados | 0.00 | **0.88** |
| pct_discapacidad | 0.12 | 0.43 |
| tasa_desercion_pct | 0.00 | **2.25** |
| estudiantes_por_escuela | **1.19** | 0.74 |

**NMF1.** El atributo de mayor peso es `estudiantes_por_escuela` (1.19), que concentra cerca
del 87% del peso del factor. Los demás atributos casi no participan (`pct_discapacidad` 0.12,
`pct_desaprobados` 0.06, `pct_retirados` y `tasa_desercion_pct` en 0.00). Por tanto, NMF1
representa un perfil de **tamaño de las escuelas**.

**NMF2.** Los atributos de mayor peso son `tasa_desercion_pct` (2.25), `pct_desaprobados`
(1.69) y `pct_retirados` (0.88), que en conjunto suman cerca del 80% del factor.
`estudiantes_por_escuela` (0.74) y `pct_discapacidad` (0.43) participan en menor medida.
Por tanto, NMF2 representa un perfil de **movilidad/salida escolar**.

Así, NMF separa dos dimensiones: el tamaño de la escuela (NMF1) y el riesgo de salida escolar
(NMF2). Todos los pesos son no negativos, por lo que cada factor se interpreta como un perfil
que se suma, no como un contraste.

In [ ]:
# 4. Representación W
W_df = pd.DataFrame(W, columns=["NMF1", "NMF2"], index=matriz_analitica.index)
W_plot = W_df.assign(Nivel=nivel)

ax = sns.scatterplot(data=W_plot, x="NMF1", y="NMF2", hue="Nivel", s=30, alpha=0.6)
ax.set_title("Representación NMF de los distritos × nivel")
ax.figure.set_size_inches(7, 5)

**Representación NMF.** En el plano NMF1-NMF2 los distritos × nivel forman una nube
continua. Aunque `nivel_bin` no se usó para construir los factores, la primaria se concentra
en valores bajos de NMF1 (tamaño de escuela) y de NMF2 (salida escolar y trayectoria débil),
mientras que la secundaria se desplaza hacia valores altos de NMF1 y alcanza los mayores
valores de NMF2. La separación entre niveles es más nítida que en PCA y ocurre sobre todo a lo
largo de NMF1, lo que refleja la diferencia de tamaño entre las escuelas de ambos niveles.
Persiste una zona de superposición y no se observan grupos claramente separados. Una franja de
puntos con NMF2 igual a 0 corresponde a casos sin participación del perfil de movilidad escolar.

**8.4 Comparar ambas técnicas**

1. ¿PCA y NMF resaltan los mismos atributos?

En parte. Ambas técnicas identifican el mismo bloque de atributos de salida escolar: tasa_desercion_pct, pct_retirados y pct_desaprobados. En PCA este bloque domina Z1 (0.61, 0.53 y 0.43) y en NMF domina NMF2 (2.25, 0.88 y 1.69).

Hay dos diferencias importantes:

Tamaño de escuela: en PCA, estudiantes_por_escuela (0.40) queda mezclado con la salida escolar dentro de Z1. En NMF aparece casi solo en NMF1 (1.19), separado de la movilidad. NMF distingue claramente “tamaño” de “riesgo de salida”.
Discapacidad: en PCA, pct_discapacidad define Z2 (0.83), con un eje propio. En NMF casi no pesa (0.12 en NMF1 y 0.43 en NMF2), así que queda como atributo secundario. PCA la resalta porque aporta varianza distinta a la del resto, y NMF solo ve lo que se suma a un perfil.

2. ¿Cuál es más fácil de interpretar?

NMF. Todos sus pesos son no negativos, así que cada factor se lee como un perfil que se suma (“esto es tamaño”, “esto es movilidad”), sin tener que interpretar signos. En PCA los pesos pueden ser positivos o negativos (por ejemplo, Z2 contrasta pct_discapacidad +0.83 contra estudiantes_por_escuela −0.43). Por eso cada componente se lee como un contraste, que es más difícil de explicar a un usuario como MINEDU o una UGEL. Además, el NMF1 y el NMF2 se pueden nombrar con claridad. En cambio, Z1 mezcla salida escolar con tamaño de escuela.

3. ¿Cuál es más útil para el clustering?

Para el objetivo del proyecto (segmentar distritos según movilidad y salida escolar), NMF resulta más útil con solo 2 dimensiones. NMF2 aísla el eje de movilidad y NMF1 el de tamaño, así que los clusters resultantes se pueden etiquetar de forma directa (por ejemplo, escuelas grandes con alta movilidad frente a escuelas pequeñas con baja movilidad). Además, la separación entre niveles se ve más nítida que en PCA.

PCA es mejor si se prioriza conservar información. Con 4 componentes retiene el 91.11% de la varianza, mientras que los 2 primeros solo acumulan 58.88%, y sus componentes no están correlacionadas, lo que favorece a K-means. Su desventaja es que 4 ejes son más difíciles de interpretar.

Ninguna de las dos proyecciones muestra grupos claramente separados, así que el clustering probablemente va a ser más una segmentación por perfiles que una separación natural. Mi recomendación sería usar NMF para clusterizar e interpretar, y PCA (4 componentes) como contraste para verificar que los grupos sean estables.

**8.5 Guardar las representaciones obtenidas**

In [ ]:
ids = matriz_analitica[["ubigeo_key", "DPTO", "PROV", "DIST", "nivel_bin"]]

Z_final = pd.concat([ids, pd.DataFrame(Z[:, :n80], columns=nombres_z[:n80], index=ids.index)], axis=1)
W_final = pd.concat([ids, W_df], axis=1)

Z_final.to_csv("pca_Z.csv", index=False, encoding="utf-8-sig")
W_final.to_csv("nmf_W.csv", index=False, encoding="utf-8-sig")
print("Z:", Z_final.shape, "| W:", W_final.shape)

**Exportar la base integrada y matriz analítica**

In [ ]:

base_final.to_csv("base_integrada_educacion.csv", index=False, encoding="utf-8-sig")
print("Archivo generado: base_integrada_educacion.csv")
print("Filas:", base_final.shape[0], "| Columnas:", base_final.shape[1])

matriz_analitica.to_csv("matriz_analitica_final.csv", index=False, encoding="utf-8-sig")
print("Archivo generado: matriz_analitica.csv")
print("Filas:", matriz_analitica.shape[0], "| Columnas:", matriz_analitica.shape[1])


diccionario_base_final.to_csv("diccionario_base_final.csv", index=False, encoding="utf-8-sig")
print("Archivo generado: diccionario_base_final.csv")
print("Filas:", diccionario_base_final.shape[0], "| Columnas:", diccionario_base_final.shape[1])

# **9. Distancias, escalamiento y K-means**

---

### <font color=green>A. Preparación</font>

**1. Unidad de análisis:** distrito × nivel educativo (primaria o secundaria). Cada fila de la matriz representa un distrito en un nivel y se identifica con `ubigeo_key` + `nivel_bin`. Un mismo distrito puede aparecer dos veces, una como primaria y otra como secundaria.

**2. Variables en la distancia:** `pct_desaprobados`, `pct_retirados`, `pct_discapacidad`, `tasa_desercion_pct` y `estudiantes_por_escuela`.

**3. Excluidas:**
- Identificadores (`ubigeo_key`, `DPTO`, `PROV`, `DIST`), porque no miden similitud.
- `nivel_bin`, porque si entra en la distancia K-means tenderá a separar primaria de secundaria y ocultará otros patrones. Se conserva solo para interpretar los grupos.
- Totales absolutos, porque el análisis compara tasas y proporciones, no volúmenes.

No hay clase externa real en el proyecto.




**4. Revisión de escalas y outliers.**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")
# Variables que participarán en el clustering.
# Excluimos identificadores y la categoría externa.
features = ["pct_desaprobados", "pct_retirados", "pct_discapacidad",
            "tasa_desercion_pct", "estudiantes_por_escuela"]

X = matriz_analitica[features].copy()

display(X.describe().T.round(3))
print("Nulos:", X.isna().sum().sum())
print("\nSesgo (skewness):")
display(X.skew().round(2))

**Boxplots por variable (cada una con su propio eje, por la diferencia de escalas)**

In [ ]:
X_long = X.melt(var_name="atributo", value_name="valor")

g = sns.catplot(
    data=X_long,
    x="atributo",
    y="valor",
    kind="box",
    col="atributo",
    sharey=False,
    sharex=False,
    height=3,
    aspect=0.8,
    col_wrap=5)
g.set_titles("{col_name}")
g.fig.suptitle("Escalas y valores extremos antes de escalar", y=1.05)
plt.show()

Las cinco variables están en escalas muy distintas: tres son proporciones entre 0 y 0.44, `tasa_desercion_pct` llega a cerca de 18.6 y `estudiantes_por_escuela` supera los 900. Todas muestran asimetría a la derecha (sesgo entre 1.88 y 3.94): la mayoría de distritos × nivel tiene valores bajos y hay una cola larga de valores altos. Estos valores corresponden a distritos genuinamente distintos (urbanos con escuelas grandes, o distritos pequeños con tasas altas) y no a errores de datos, por lo que se conservan. Escalar no los corrige, así que pueden influir en los centroides y se señala como limitación.


**5. Escalamiento elegido.**

 K-means usa distancia euclidiana, por lo que una variable con valores grandes domina la distancia solo por su escala (`estudiantes_por_escuela` y `tasa_desercion_pct` pesaría mucho más que las proporciones).

 Primero se aplica `log1p` a las cinco variables para reducir su fuerte asimetría (las proporciones se expresan antes en %, porque `log1p` sobre valores entre 0 y 1 casi no modifica nada).

 Luego se aplica `StandardScaler` (z-score) para que las cinco tengan media 0 y desviación estándar 1. Se prefiere z-score sobre Min-Max porque, con tantos valores extremos, Min-Max comprimiría a la mayoría de los distritos en un rango estrecho.
 Los z-scores negativos son válidos: indican valores por debajo de la media.

In [ ]:
X_t = X.copy()

# Las proporciones (0 a 1) se pasan a % para que el log1p sí haga efecto
for c in ["pct_desaprobados", "pct_retirados", "pct_discapacidad"]:
    X_t[c] = np.log1p(100 * X_t[c])

X_t["tasa_desercion_pct"] = np.log1p(X_t["tasa_desercion_pct"])
X_t["estudiantes_por_escuela"] = np.log1p(X_t["estudiantes_por_escuela"])

print("Sesgo después de la transformación:")
display(X_t.skew().round(2))

scaler = StandardScaler()
X_std = scaler.fit_transform(X_t)

X_std_df = pd.DataFrame(X_std, columns=features, index=matriz_analitica.index)
print("\nVariables estandarizadas:")
display(X_std_df.describe().T.round(2))

In [ ]:
X_std_df.head()

Tras la transformación, el sesgo bajó de entre 1.88 y 3.94 a valores entre -0.12 y 1.38. Cuatro de las cinco variables quedaron con sesgo menor a 1; `pct_retirados` es la que conserva más asimetría (1.38), porque más de la mitad de los distritos × nivel tiene valores muy cercanos a 0 (su percentil 25 es exactamente 0 y su mediana es 0.002). Tras el z-score, las cinco variables tienen media 0 y desviación estándar 1.

### <font color=green>B. K-means</font>

6. Evalúen al menos `K = 1, 2, 3, 4, 5, 6`.
7. Presenten el gráfico de **inercia vs. K**.
8. Propongan un `K` tentativo mediante el **método del codo**.
9. Apliquen K-means con ese valor y reporten el tamaño de cada cluster.
10. Perfilen los clusters usando las variables en sus **unidades originales**.

**6. Evalúen al menos `K = 1, 2, 3, 4, 5, 6`.**

In [ ]:
# Lista donde guardamos K y la inercia de cada solución.
resultados_k = []

for k in range(1, 7):   # incluye K = 1, 2, 3, 4, 5, 6 (y 7)

    modelo = KMeans(
        n_clusters=k,
        init="k-means++",    # centroides iniciales bien separados
        n_init=10,           # 10 ejecuciones, se conserva la mejor
        random_state=42      # resultados reproducibles
    )

    modelo.fit(X_std_df)

    resultados_k.append({
        "k": k,
        "inercia": modelo.inertia_
    })

resultados_k = pd.DataFrame(resultados_k)

display(resultados_k.round(2))

**7.  Presenten el gráfico de inercia vs. K**

In [ ]:
ax = sns.lineplot(
    data=resultados_k,
    x="k",
    y="inercia",
    marker="o"
)

ax.set(
    title="Método del codo",
    xlabel="Número de clusters (K)",
    ylabel="Inercia"
)

plt.show()

**8. Propongna un K tentativo mediante el método del codo**



In [ ]:
# K tentativo según el método del codo.

k_elegido = 2
print("K tentativo elegido:", k_elegido)

Según el método del codo, se propone **K = 2** de forma tentativa: es el punto donde la inercia cae más (28.3% respecto a K = 1) y desde K = 3 las mejoras son menores (13.5% o menos). El detalle de las reducciones se muestra en la parte C.

**9. Apliquen K-means con ese valor y reporten el tamaño de cada cluster.**

In [ ]:
# K tentativo.
k_elegido = 2

kmeans = KMeans(
    n_clusters=k_elegido,
    init="k-means++",
    n_init=10,
    random_state=42
)

# Ajustamos K-means y obtenemos la etiqueta de cada observación.
# La columna se agrega a la matriz analítica (producto C).
matriz_analitica["cluster_kmeans"] = kmeans.fit_predict(X_std)

print("Inercia:", round(kmeans.inertia_, 2))

print("\nTamaño de los clusters:")
display(matriz_analitica["cluster_kmeans"].value_counts().sort_index())

In [ ]:
ax = sns.countplot(data=matriz_analitica, x="cluster_kmeans")
ax.bar_label(ax.containers[0])
ax.set(title="Tamaño de cada cluster", xlabel="Cluster", ylabel="Distritos × nivel")
plt.show()

El gráfico muestra el tamaño de cada grupo identificado por el algoritmo K-Means. Se observa que el cluster 0 tiene 2201 registros, mientras que el cluster 1 tiene 1529 registros.
Esto significa que el cluster 0 contiene una mayor cantidad de distritos o registros analizados que el cluster 1. Por otro lado, el cluster 1 es el grupo más pequeño.

**10. Perfilen los clusters usando las variables en sus unidades originales.**

In [ ]:
# Promedio de cada variable ORIGINAL (sin log ni z-score) dentro de cada cluster.
perfil = (
    matriz_analitica.groupby("cluster_kmeans")[features]
      .mean()
      .round(4)
)

# Tamaño de cada cluster.
tamanos = (
    matriz_analitica.groupby("cluster_kmeans")
      .size()
      .rename("n")
)

perfil = perfil.join(tamanos)

display(perfil)

El cluster 0 (n = 2,201) presenta menores niveles de desaprobación, retiro y deserción escolar, además de un menor número de estudiantes por escuela. En cambio, el cluster 1 (n = 1,529) presenta indicadores más altos en los tres y escuelas más grandes, y es el grupo más pequeño.

In [ ]:
# Pasamos los promedios de cada cluster a z-scores,
# usando la media y desviación de los datos originales.
# NO cambia la tabla anterior: solo facilita la comparación visual.
perfil_z = perfil[features].copy()
perfil_z = (perfil_z - X.mean()) / X.std(ddof=0)

plt.figure(figsize=(9, 3))

sns.heatmap(
    perfil_z,
    annot=True,
    fmt=".2f",
    center=0,
    cmap="vlag",
    linewidths=0.5
)

plt.title("Perfil relativo de los clusters")
plt.xlabel("Atributos")
plt.ylabel("Cluster")
plt.show()


Cluster 0: Movilidad escolar baja

Cluster 1: Movilidad  escolar alta

### <font color=green>C. Producto para el proyecto</font>

- Agreguen una columna `cluster_kmeans` a su matriz analítica.
- Presenten el gráfico del codo.
- Incluyan una tabla de perfilamiento.
- Escriban una conclusión de máximo **6 líneas** explicando qué caracteriza a cada grupo y qué limitaciones tiene todavía la solución.



**Agregar una columna cluster_kmeans a la matriz analitica**

In [ ]:
print(matriz_analitica[["ubigeo_key", "DIST", "nivel_bin", "cluster_kmeans"]].head())
matriz_analitica.to_csv("matriz_analitica_con_clusters.csv", index=False, encoding="utf-8-sig")

In [ ]:
matriz_analitica

**Presentar el gráfico de codo**

In [ ]:
# Reducción porcentual respecto al K anterior (K = 1 no tiene anterior).
resultados_k["reduccion_pct"] = (-100 * resultados_k["inercia"].pct_change()).round(1)

ax = sns.lineplot(
    data=resultados_k,
    x="k",
    y="inercia",
    marker="o"
)

# Escribimos el porcentaje sobre cada punto, desde K = 2.
for _, fila in resultados_k.iterrows():
    if pd.notna(fila["reduccion_pct"]):
        ax.annotate(
            f"-{fila['reduccion_pct']}%",
            (fila["k"], fila["inercia"]),
            textcoords="offset points",
            xytext=(12, 8),
            ha="center"
        )

ax.set(
    title="Método del codo",
    xlabel="Número de clusters (K)",
    ylabel="Inercia"
)

plt.show()


La inercia disminuye de forma continua al aumentar K. La mayor reducción ocurre al pasar de K = 1 a K = 2 (28.3%); después la caída se reduce a 13.5% (K = 3), 12.9% (K = 4), 8.5% (K = 5) y 8.6% (K = 6), y la curva se aplana gradualmente sin un codo muy marcado. Se propone K = 2 de forma tentativa, por ser el punto de mayor mejora; K = 3 también sería razonable y se evaluará con Silhouette y Davies-Bouldin en la siguiente sesión.

**Evaluación de K con tres métricas: inercia, Silhouette y Davies-Bouldin**

El codo de la inercia no era marcado, y en la interpretación anterior quedó pendiente evaluar K = 3 con Silhouette y Davies-Bouldin. Se calculan las tres métricas para K = 2 a 8 sobre la misma matriz escalada (`X_std_df`):

| Métrica | Qué mide | Cómo se elige K |
|---|---|---|
| Inercia | Dispersión dentro de los clusters | Codo (donde deja de bajar con fuerza) |
| Silhouette | Qué tan separado y compacto es cada caso respecto a su cluster | **Máximo** |
| Davies-Bouldin | Parecido entre clusters (solapamiento) | **Mínimo** |

Silhouette y Davies-Bouldin necesitan al menos 2 clusters, por eso K = 1 no se evalúa.

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

rango_k = range(2, 9)

filas = []
for k in rango_k:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    etiquetas = km.fit_predict(X_std_df)          # misma matriz escalada de K-means
    filas.append({
        "k": k,
        "inercia": km.inertia_,
        "silhouette": silhouette_score(X_std_df, etiquetas),
        "davies_bouldin": davies_bouldin_score(X_std_df, etiquetas),
    })
metricas_k = pd.DataFrame(filas)
display(metricas_k.round(3))

# K sugerido por cada criterio
k_sil = int(metricas_k.loc[metricas_k["silhouette"].idxmax(), "k"])          # pico
k_db  = int(metricas_k.loc[metricas_k["davies_bouldin"].idxmin(), "k"])      # valle
print(f"K por Silhouette (máximo): {k_sil} | K por Davies-Bouldin (mínimo): {k_db} | K elegido con el codo: {k_elegido}")

config = [
    ("inercia",        "Inercia",          "Menor = mejor (buscar el codo)",  "#5B4FC4", k_elegido),
    ("silhouette",     "Silhouette score", "Mayor = mejor (buscar el pico)",  "#1B7F5F", k_sil),
    ("davies_bouldin", "Davies-Bouldin",   "Menor = mejor (buscar el valle)", "#D9622B", k_db),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, (col, titulo, sub, color, k_mejor) in zip(axes, config):
    ax.plot(metricas_k["k"], metricas_k[col], marker="o", color=color)
    ax.fill_between(metricas_k["k"], metricas_k[col], metricas_k[col].min(), color=color, alpha=0.1)
    y = metricas_k.loc[metricas_k["k"] == k_mejor, col].iloc[0]       # punto rojo en el K marcado
    ax.scatter([k_mejor], [y], s=140, color="red", zorder=3)
    ax.set_title(f"{titulo}\n{sub}", fontsize=11)
    ax.set_xlabel("K")
    ax.set_xticks(list(rango_k))
fig.suptitle("Elección de K con inercia, Silhouette y Davies-Bouldin (punto rojo = K marcado por cada criterio)", y=1.03)
plt.tight_layout()
plt.show()

# Conclusión automática
if k_sil == k_db == k_elegido:
    print(f"Lectura automática: las tres métricas coinciden en K = {k_elegido}.")
elif k_sil == k_db:
    print(f"Lectura automática: Silhouette y Davies-Bouldin coinciden en K = {k_sil}, distinto del K = {k_elegido} del codo.")
else:
    print(f"Lectura automática: las métricas no coinciden (Silhouette: K = {k_sil}; Davies-Bouldin: K = {k_db}; codo: K = {k_elegido}).")

**Interpretación:** esta evaluación responde a lo que quedó pendiente en el análisis del codo (¿es mejor K = 2 o K = 3?). La línea impresa debajo del gráfico dice si las tres métricas coinciden con el K = 2 elegido.

- Si **coinciden**, K = 2 queda respaldado por tres criterios independientes y no solo por el codo, que no era marcado.
- Si **Silhouette y Davies-Bouldin apuntan a otro K**, hay que decidir con criterio de uso: K = 2 sigue siendo más fácil de interpretar para el usuario (MINEDU/DRE/UGEL: salida alta vs baja), mientras que un K mayor separaría perfiles intermedios.
- En cualquier caso, los valores de Silhouette de esta tabla son **bajos** (referencia: K = 2 tiene 0.265 y Davies-Bouldin 1.465), coherente con que los distritos forman una nube continua y los grupos son una simplificación útil, no categorías nítidas.

**Tabla de perfilamiento**

In [ ]:
perfil = (
    matriz_analitica.groupby("cluster_kmeans")[features]
      .mean()
      .round(4)
)

tamanos = matriz_analitica.groupby("cluster_kmeans").size().rename("n")

perfil = perfil.join(tamanos)

display(perfil)

**Conclusión maximo 6 lineas explicando qué caracteriza a cada grupo y qué limitaciones tiene todavía la solución.**


K-means con K = 2 separó los 3,730 distritos × nivel en dos grupos. El cluster 0 (2,201 casos, 59.0%) reúne escuelas pequeñas (unos 62 estudiantes) con baja desaprobación (2.7%), retiro (0.19%) y deserción (0.50%). El cluster 1 (1,529 casos, 41.0%) agrupa escuelas más grandes (unos 150 estudiantes) con mayor desaprobación (5.6%), retiro (1.95%) y deserción (2.9%); `pct_discapacidad` casi no diferencia los grupos. Limitaciones: el codo no es marcado (K = 3 también es razonable), los outliers y los ceros en retiro y deserción pueden influir en los centroides, y K-means asume grupos esféricos de tamaño similar. Se reevaluará con Silhouette y Davies-Bouldin.

# **10. DBSCAN**

---

## <font color=green>A. DBSCAN</font>

**1. Matriz con la que se trabaja.** Se usa la **misma representación de K-means**: `X_std_df` (5 variables con `log1p` + z-score). No entran `nivel_bin`, identificadores ni `cluster_kmeans`.

In [ ]:
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Matriz escalada de K-means (sin cluster_kmeans, sin nivel_bin).
X_db = X_std_df.values
print("Forma de la matriz:", X_db.shape)
print("Variables:", list(X_std_df.columns))

**2. Valor inicial de `min_samples`.** Se propone `min_samples = 10`.

- Una regla práctica es usar al menos `2 × (número de variables)` = 2 × 5 = 10.
- Con 3 730 observaciones, un valor muy bajo (por ejemplo 3 o 5) haría que casi cualquier punto sea núcleo y se formarían zonas densas por casualidad.
- Un valor mayor exige más vecinos y vuelve más estricta la definición de "zona densa".

Recordar que el conteo incluye al propio punto.

**3. Gráfico de k-distancias.** Para cada observación se calcula la distancia a su vecino número `min_samples`, se ordenan y se busca el punto donde la curva empieza a subir con rapidez.

In [ ]:
min_samples = 10

# Cada punto se cuenta a sí mismo, igual que DBSCAN.
vecinos = NearestNeighbors(n_neighbors=min_samples).fit(X_db)
distancias, _ = vecinos.kneighbors(X_db)

# Distancia al último vecino considerado, ordenada de menor a mayor.
k_dist = np.sort(distancias[:, -1])

kdist_df = pd.DataFrame({
    "observacion_ordenada": np.arange(len(k_dist)),
    "k_distancia": k_dist
})

# Ayuda para ubicar el codo: punto más alejado de la diagonal
# cuando ambos ejes se llevan a la escala [0, 1].
xn = kdist_df["observacion_ordenada"] / kdist_df["observacion_ordenada"].max()
yn = (k_dist - k_dist.min()) / (k_dist.max() - k_dist.min())
idx_codo = int(np.argmax(xn - yn))
eps_codo = float(k_dist[idx_codo])

ax = sns.lineplot(data=kdist_df, x="observacion_ordenada", y="k_distancia")
ax.axhline(eps_codo, color="red", linestyle="--")
ax.text(0, eps_codo, f"  codo aprox. = {eps_codo:.2f}", color="red", va="bottom")
ax.set(
    title=f"Gráfico de {min_samples}-distancias",
    xlabel="Observaciones ordenadas",
    ylabel=f"Distancia al vecino {min_samples}"
)
plt.show()

print("Percentiles de la k-distancia:")
display(pd.Series(k_dist).quantile([0.50, 0.75, 0.90, 0.95, 0.99]).round(3).to_frame("k_distancia"))

**4. Valor inicial de `eps`.** El codo marcado en el gráfico es solo un **punto de partida**, no una regla automática. Revisar el gráfico y, si el codo visual está en otro lugar, ajustar `eps_base`.

In [ ]:
# Valor inicial de eps tomado del codo. Si al mirar el gráfico se ve otro codo, cambiarlo aquí.
eps_base = round(eps_codo, 2)
print("eps inicial propuesto:", eps_base)

**5. Tres valores cercanos de `eps`.** Se mantiene fijo `min_samples = 10` y se observa cuánto cambia la solución.

In [ ]:
# Tres valores cercanos: uno menor, el inicial y uno mayor.
eps_values = [round(eps_base * 0.8, 2), eps_base, round(eps_base * 1.25, 2)]

filas = []
etiquetas_por_eps = {}

for eps in eps_values:
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_db)
    etiquetas_por_eps[eps] = labels

    conteos = pd.Series(labels).value_counts().sort_index()
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    filas.append({
        "eps": eps,
        "n_clusters": n_clusters,
        "ruido_%": round(100 * np.mean(labels == -1), 1),
        "tamaños (etiqueta: n)": dict(conteos)   # -1 = ruido
    })

comparacion_eps = pd.DataFrame(filas)
display(comparacion_eps)

In [ ]:
# Visualización solo para ver los grupos: PCA a 2 dimensiones (no se usa para el clustering).
Z2 = PCA(n_components=2, random_state=42).fit_transform(X_db)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True, sharey=True)
for ax, eps in zip(axes, eps_values):
    viz = pd.DataFrame({"Z1": Z2[:, 0], "Z2": Z2[:, 1],
                        "cluster": pd.Series(etiquetas_por_eps[eps]).astype("category")})
    sns.scatterplot(data=viz, x="Z1", y="Z2", hue="cluster", s=12, alpha=0.7, ax=ax)
    ax.set_title(f"DBSCAN — eps={eps}, min_samples={min_samples}")
plt.tight_layout()
plt.show()

**6. Solución elegida y resultados.** Se elige un `eps` de los tres probados (cambiar `eps_final` según lo observado: el que dé una estructura interpretable sin dejar demasiado ruido ni un único grupo gigante). Se reportan número de clusters, porcentaje de ruido y tamaño de cada grupo.

In [ ]:
# Elegir uno de los tres valores probados (ajustar según lo observado en la tabla y los gráficos).
eps_final = eps_values[1]

dbscan = DBSCAN(eps=eps_final, min_samples=min_samples)
labels_dbscan = dbscan.fit_predict(X_db)

# 7. Guardar las etiquetas (-1 = ruido).
matriz_analitica["cluster_dbscan"] = labels_dbscan

n_clusters_db = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
print(f"eps = {eps_final} | min_samples = {min_samples}")
print("Número de clusters (sin ruido):", n_clusters_db)
print("Porcentaje de ruido:", round(100 * np.mean(labels_dbscan == -1), 1), "%")
print("\nTamaño por etiqueta (-1 = ruido):")
display(matriz_analitica["cluster_dbscan"].value_counts().sort_index().to_frame("n"))

In [ ]:
# Perfil de cada grupo DBSCAN en unidades originales (incluye el ruido como -1).
perfil_db = (
    matriz_analitica.groupby("cluster_dbscan")[features].mean().round(4)
    .join(matriz_analitica.groupby("cluster_dbscan").size().rename("n"))
)
display(perfil_db)

**Lectura de DBSCAN.** Con `min_samples = 10` y `eps = 1.06`, DBSCAN encontró un único cluster (3 563 casos, 95.5 %) y marcó 167 casos como ruido (4.5 %). El resultado es muy sensible a `eps`: con 0.85 el ruido sube a 11.3 % y aparece un segundo grupo de solo 7 casos; con 1.33 el ruido baja a 1.0 % y queda un solo grupo. Esto es coherente con lo visto en PCA y NMF: los datos forman una nube continua, sin zonas densas separadas por huecos. Los puntos de ruido tienen más retiro (4.1 % vs 0.8 %), discapacidad (4.6 % vs 1.8 %) y deserción (3.8 % vs 1.4 %), escuelas más pequeñas (68 vs 99 estudiantes) y 73 % de secundaria. Ruido no es un error, pero parte de estos casos podría ser distritos pequeños con porcentajes inestables (por verificar).

## <font color=green>B. Clustering jerárquico</font>

**8. Dendrograma con Ward.** Con 3 730 observaciones el árbol completo sería ilegible, así que se dibuja con una **muestra reproducible de 300 observaciones solo para explorar**. La partición final se calcula después con todos los datos.

In [ ]:
# Muestra reproducible SOLO para visualizar el dendrograma.
muestra = X_std_df.sample(n=300, random_state=42)

Z_ward_m = linkage(muestra.values, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z_ward_m, no_labels=True)
plt.title("Dendrograma aglomerativo — Ward (muestra de 300 observaciones)")
plt.xlabel("Observaciones (muestra)")
plt.ylabel("Distancia de fusión")
plt.show()

**10. Otro criterio de enlace.** Se compara Ward con `complete` y `average` sobre la misma muestra.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, metodo in zip(axes, ["ward", "complete", "average"]):
    Z_m = linkage(muestra.values, method=metodo)
    dendrogram(Z_m, no_labels=True, ax=ax)
    ax.set_title(f"linkage = {metodo}")
    ax.set_ylabel("Distancia de fusión")
plt.tight_layout()
plt.show()

**11. Dónde cortar el árbol.** Con todos los datos se calcula la jerarquía con Ward y se miran las últimas fusiones: un salto grande en la altura sugiere que se están uniendo grupos relativamente distintos, y por tanto conviene cortar justo antes.

In [ ]:
# Jerarquía Ward con todos los datos (solo para decidir el corte).
Z_ward = linkage(X_db, method="ward")

alturas = Z_ward[:, 2]
filas = []
for k in range(2, 7):   # se consideran 2 a 6 clusters, como en K-means
    # Pasar de k a k-1 clusters ocurre en la altura alturas[-(k-1)];
    # la fusión anterior ocurre en alturas[-k].
    altura_fusion = alturas[-(k - 1)]
    altura_previa = alturas[-k]
    filas.append({
        "k (clusters que se conservan)": k,
        "altura al fusionar k→k-1": round(altura_fusion, 2),
        "salto respecto a la fusión anterior": round(altura_fusion - altura_previa, 2)
    })
tabla_saltos = pd.DataFrame(filas)
display(tabla_saltos)

# Corte propuesto: el mayor salto. Confirmarlo con el dendrograma; se puede cambiar manualmente (por ejemplo k_hier = 3).
k_hier = int(tabla_saltos.loc[tabla_saltos["salto respecto a la fusión anterior"].idxmax(),
                              "k (clusters que se conservan)"])
print("k propuesto por el mayor salto:", k_hier)

In [ ]:
# Efecto del criterio de enlace con el mismo número de clusters.
resumen_linkage = []
etiquetas_linkage = {}
for metodo in ["ward", "complete", "average", "single"]:
    lab = AgglomerativeClustering(n_clusters=k_hier, linkage=metodo).fit_predict(X_db)
    etiquetas_linkage[metodo] = lab
    conteos = pd.Series(lab).value_counts()
    resumen_linkage.append({
        "linkage": metodo,
        "n_clusters": len(conteos),
        "tamaño_min": conteos.min(),
        "tamaño_max": conteos.max()
    })
display(pd.DataFrame(resumen_linkage))

**Lectura del criterio de enlace:** con el mismo número de clusters (2), **Ward** y **complete** dan grupos balanceados (1 155 / 2 575 y 1 506 / 2 224 casos). **Average** y **single** producen un grupo de 3 727 y 3 729 casos y otro de 3 y 1: es el *efecto de encadenamiento*, que ocurre cuando los datos forman una nube continua sin huecos. Por eso se conserva Ward, que además minimiza la varianza dentro de cada grupo.

In [ ]:
# Partición final: Ward con k_hier clusters sobre todos los datos.
labels_hier = AgglomerativeClustering(n_clusters=k_hier, linkage="ward").fit_predict(X_db)

# 11. Guardar las etiquetas.
matriz_analitica["cluster_hier"] = labels_hier

print("Tamaño de cada cluster (Ward):")
display(matriz_analitica["cluster_hier"].value_counts().sort_index().to_frame("n"))

# Perfil en unidades originales.
perfil_hier = (
    matriz_analitica.groupby("cluster_hier")[features].mean().round(4)
    .join(matriz_analitica.groupby("cluster_hier").size().rename("n"))
)
display(perfil_hier)

# Visualización en el plano PCA (solo para ver los grupos).
viz = pd.DataFrame({"Z1": Z2[:, 0], "Z2": Z2[:, 1],
                    "cluster": pd.Series(labels_hier).astype("category")})
ax = sns.scatterplot(data=viz, x="Z1", y="Z2", hue="cluster", s=15, alpha=0.7)
ax.set_title(f"Clustering jerárquico — Ward, k={k_hier}")
plt.show()

## <font color=green>C. Comparación de las tres soluciones</font>

In [ ]:
# Tablas cruzadas: ¿cómo se reparten los casos entre los métodos?
print("K-means vs Jerárquico:")
display(pd.crosstab(matriz_analitica["cluster_kmeans"], matriz_analitica["cluster_hier"],
                    rownames=["kmeans"], colnames=["hier"]))

print("K-means vs DBSCAN (-1 = ruido):")
display(pd.crosstab(matriz_analitica["cluster_kmeans"], matriz_analitica["cluster_dbscan"],
                    rownames=["kmeans"], colnames=["dbscan"]))

# nivel_bin NO se usó en ningún clustering; solo sirve para interpretar.
print("Composición por nivel (0 = Primaria, 1 = Secundaria), en %:")
for col in ["cluster_kmeans", "cluster_dbscan", "cluster_hier"]:
    print("\n", col)
    display((100 * pd.crosstab(matriz_analitica[col], matriz_analitica["nivel_bin"],
                               normalize="index")).round(1))

In [ ]:
# Guardar las tres etiquetas en la matriz analítica.
cols_cluster = ["cluster_kmeans", "cluster_dbscan", "cluster_hier"]
display(matriz_analitica[["ubigeo_key", "DIST", "nivel_bin"] + cols_cluster].head())

matriz_analitica.to_csv("matriz_analitica_con_clusters.csv", index=False, encoding="utf-8-sig")
print("Guardado: matriz_analitica_con_clusters.csv")

### Conclusión grupal — máximo 8 líneas

Respondan:

- ¿Qué diferencias importantes observan entre las tres soluciones?
- ¿Algún método parece inadecuado para la estructura de sus datos? ¿Por qué?
- ¿Qué decisión de preprocesamiento o parámetros parece influir más?

> **No es obligatorio que los tres métodos produzcan una buena solución.**  
> La próxima semana utilizaremos Silhouette, Davies-Bouldin y perfilamiento para evaluar e interpretar con mayor rigor los resultados.

**Guarden las tres etiquetas:** `cluster_kmeans`, `cluster_dbscan` y `cluster_hier`.


**Conclusión grupal**

K-means (K = 2) y Ward (k = 2) dan soluciones parecidas (coinciden en el 84.6 % de los casos): Salida escolar baja (Ward: 2 575 casos, deserción 0.80 %, 79 estudiantes por escuela) y otro salida escolar alta (1 155 casos, 2.99 %, 140 estudiantes). DBSCAN (eps = 1.06) no segmenta: forma un solo cluster (95.5 %) y marca 4.5 % de ruido, útil para detectar casos extremos (más retiro, discapacidad y deserción; 73 % secundaria) pero no para definir perfiles.
**Método menos adecuado:** DBSCAN, porque los datos forman una nube continua sin huecos y su resultado cambia mucho con `eps` (2 clusters y 11.3 % de ruido con 0.85; 1 cluster y 1.0 % con 1.33).
**Lo que más influye:** el valor de `eps` en DBSCAN, el criterio de enlace en el jerárquico (average y single dejan un grupo de 3 727 a 3 729 casos por encadenamiento; ward y complete dan grupos balanceados) y el escalamiento con `log1p` + z-score, que define el peso de cada variable en la distancia. `nivel_bin` no se usó, pero el grupo de mayor salida tiene mucha más secundaria (K-means 77 %, Ward 90 %), así que parte de la estructura refleja la diferencia entre niveles.

## <font color=green>C.2 Gráficos de apoyo a la comparación</font>

Gráficos adicionales para sustentar la conclusión grupal: los tres métodos en el mismo plano, perfiles, coincidencias, calidad por observación, composición por nivel y corte del dendrograma.

**1. Los tres métodos en el mismo plano PCA** (el PCA solo sirve para dibujar; los clusters se calcularon con las 5 variables escaladas).

In [ ]:
pca_viz = PCA(n_components=2, random_state=42)
Zp = pca_viz.fit_transform(X_db)

paneles = [
    ("K-means (K = 2)",                 "cluster_kmeans"),
    (f"DBSCAN (eps = {eps_final})",     "cluster_dbscan"),
    (f"Ward (k = {k_hier})",            "cluster_hier"),
]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharex=True, sharey=True)
for ax, (titulo, col) in zip(axes, paneles):
    etiquetas = matriz_analitica[col].astype(int).values
    viz = pd.DataFrame({"Z1": Zp[:, 0], "Z2": Zp[:, 1], "cluster": etiquetas.astype(str)})
    orden = [str(c) for c in sorted(np.unique(etiquetas))]
    sns.scatterplot(data=viz, x="Z1", y="Z2", hue="cluster", hue_order=orden,
                    s=12, alpha=0.7, ax=ax)
    ax.set_title(titulo)
    ax.set_xlabel(f"Z1 ({100 * pca_viz.explained_variance_ratio_[0]:.1f}% var.)")
    ax.set_ylabel(f"Z2 ({100 * pca_viz.explained_variance_ratio_[1]:.1f}% var.)")
plt.suptitle("Comparación de las tres soluciones en el plano PCA (DBSCAN: −1 = ruido)", y=1.02)
plt.tight_layout()
plt.show()

**Interpretación:** los puntos forman una **nube continua** sin huecos, y ninguno de los tres métodos encuentra grupos separados visualmente. K-means corta la nube con una frontera casi vertical en Z1 (salida baja a la izquierda, alta a la derecha); Ward hace un corte parecido; DBSCAN deja un solo cluster y marca como ruido unos pocos puntos de la periferia. Dos advertencias: (1) los dos ejes explican solo cerca del 60% de la varianza, así que dos grupos que se superponen aquí pueden estar más separados en las 5 dimensiones; (2) el color de cada cluster no es comparable entre paneles (el 0 de DBSCAN no es el 0 de K-means).

**2. Perfil relativo (z-score) de cada solución.** Rojo = por encima del promedio general; azul = por debajo.

In [ ]:
def perfil_relativo(col):
    medias = matriz_analitica.groupby(col)[features].mean()
    return (medias - matriz_analitica[features].mean()) / matriz_analitica[features].std(ddof=0)

paneles_perfil = [("K-means", "cluster_kmeans"), ("DBSCAN (−1 = ruido)", "cluster_dbscan"), ("Ward", "cluster_hier")]
fig, axes = plt.subplots(1, 3, figsize=(19, 3.8))
for i, (ax, (titulo, col)) in enumerate(zip(axes, paneles_perfil)):
    sns.heatmap(perfil_relativo(col), annot=True, fmt=".2f", center=0, vmin=-2, vmax=2,
                cmap="vlag", linewidths=0.5, ax=ax, cbar=(i == 2))
    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.set_ylabel("Cluster")
    ax.tick_params(axis="x", rotation=30)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
fig.suptitle("Perfil relativo (z-score) de los clusters según el método", y=1.04)
plt.tight_layout()
plt.show()

**Interpretación:** en K-means y en Ward el cluster de **salida alta** (cluster 1) está por encima del promedio en casi todas las variables de salida escolar. En K-means, frente al cluster 0: deserción 2.90% vs 0.50%, retiro 1.95% vs 0.19%, desaprobación 5.6% vs 2.7% y escuelas más grandes (150 vs 62 estudiantes). En Ward la diferencia de deserción es 2.99% vs 0.80%. `pct_discapacidad` **no diferencia** los grupos (1.86% vs 1.91% en K-means). El **ruido de DBSCAN** (167 casos) es otro tipo de grupo: tiene deserción alta (3.82%) y mucha más discapacidad (4.6% vs 1.8%) y retiro (4.1% vs 0.8%), pero con **escuelas más pequeñas** (68 vs 99 estudiantes). Son casos extremos, no un perfil de salida generalizada.

**3. Silhouette por observación** (K-means y Ward). El ancho de cada franja es el tamaño del cluster; las barras que se van a la izquierda de 0 son casos mal asignados. La línea roja es el Silhouette promedio.

In [ ]:
from sklearn.metrics import silhouette_samples

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True)
for ax, (titulo, col) in zip(axes, [("K-means (K = 2)", "cluster_kmeans"), (f"Ward (k = {k_hier})", "cluster_hier")]):
    etiquetas = matriz_analitica[col].astype(int).values
    s = silhouette_samples(X_db, etiquetas)
    y0 = 0
    for c in np.unique(etiquetas):
        sc = np.sort(s[etiquetas == c])
        ax.fill_betweenx(np.arange(y0, y0 + len(sc)), 0, sc, alpha=0.75, label=f"cluster {c} (n = {len(sc)})")
        y0 += len(sc) + 60
    ax.axvline(s.mean(), color="red", linestyle="--")
    ax.text(s.mean(), y0 * 0.98, f" promedio = {s.mean():.3f}", color="red", va="top")
    ax.set_title(titulo)
    ax.set_xlabel("Coeficiente de silueta")
    ax.set_yticks([])
    ax.legend(loc="lower right", fontsize=8)
fig.suptitle("Silhouette por observación", y=1.02)
plt.tight_layout()
plt.show()

**Interpretación:** ambos métodos tienen una estructura débil (Silhouette promedio 0.265 en K-means y 0.238 en Ward; como referencia, valores de 0.25 a 0.5 se consideran estructura débil). K-means tiene casi todas las barras positivas y una cola negativa muy pequeña; Ward tiene una cola negativa visible en su cluster 1 (casos más cerca del otro grupo que del suyo). El cluster 0 de K-means (n = 2 201) es el más compacto, con valores cercanos a 0.5. Esto respalda elegir K-means, con la cautela de que los grupos son una simplificación de un continuo.

**4. Composición por nivel educativo de cada cluster.** `nivel_bin` no entró al clustering; este gráfico muestra cuánta de la estructura refleja la diferencia primaria/secundaria.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, (titulo, col) in zip(axes, [("K-means", "cluster_kmeans"), ("Ward", "cluster_hier")]):
    comp_nivel = (pd.crosstab(matriz_analitica[col],
                              matriz_analitica["nivel_bin"].map({0: "Primaria", 1: "Secundaria"}),
                              normalize="index") * 100)
    comp_nivel.plot(kind="bar", stacked=True, ax=ax, color=["#8FB8DE", "#264653"], edgecolor="white")
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.0f%%", label_type="center", color="white", fontsize=9)
    ax.set_title(titulo)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("% de distritos × nivel")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="", loc="upper right", fontsize=8)
fig.suptitle("Composición por nivel educativo de cada cluster", y=1.02)
plt.tight_layout()
plt.show()

**Interpretación:** el cluster de salida alta es mayormente secundaria: 77% en K-means y 90% en Ward, frente a 30% y 31% de secundaria en el cluster de salida baja. Es coherente con que la secundaria tenga más retiro y deserción, pero también significa que parte de la estructura refleja la diferencia entre niveles y no solo el comportamiento de cada distrito. Los grupos no se reducen a "primaria vs secundaria" (K-means mantiene 23% de primaria en salida alta), y Ward separa más por nivel que K-means. Por eso los mapas comparan los dos niveles por separado.

**6. Dendrograma con todos los datos y línea de corte.** Se muestran los últimos 30 grupos fusionados; la línea roja marca la altura de corte que da `k_hier` clusters.

In [ ]:
altura_corte = (Z_ward[-(k_hier - 1), 2] + Z_ward[-k_hier, 2]) / 2

fig, ax = plt.subplots(figsize=(11, 4.5))
dendrogram(Z_ward, truncate_mode="lastp", p=30, show_leaf_counts=True,
           leaf_rotation=90, leaf_font_size=8, ax=ax, color_threshold=altura_corte)
ax.axhline(altura_corte, color="red", linestyle="--")
ax.text(ax.get_xlim()[1], altura_corte, f" corte → k = {k_hier}", color="red", va="bottom", ha="right")
ax.set_title("Dendrograma Ward (todos los datos, truncado a 30 grupos)")
ax.set_xlabel("Grupos fusionados (entre paréntesis: casos en la rama)")
ax.set_ylabel("Distancia de fusión (Ward)")
plt.tight_layout()
plt.show()

**Interpretación:** la última fusión (de 2 a 1 grupo) ocurre a una altura de 91.2, con un salto de 33.6 respecto a la fusión anterior; los saltos siguientes son mucho menores (10.0, 3.1, 4.3, 5.9). Ese salto grande indica que **solo la última fusión une grupos claramente distintos**, y por eso el corte propuesto es k = 2. Cortar en k = 3 o más sería partir ramas que se parecen mucho entre sí.

# **11. Análisis geoespacial con GeoPandas (Desafío S12 aplicado al proyecto)**

Hasta aquí los clusters se interpretaron con tablas y gráficos estadísticos. Con la sesión de GeoPandas podemos
responder la pregunta que le importa al usuario (MINEDU / DRE / UGEL): **¿dónde están los distritos con mayor salida escolar y en qué nivel?**

La ubicación se usará **solo para visualizar** los resultados: ningún clustering de las secciones 9 y 10 usó coordenadas.

> **Antes de ejecutar:** subir a Colab la carpeta `peru_shapes/` (dentro de `RUTA`) con los shapefiles ADM1, ADM2 y ADM3
> (`.shp`, `.dbf`, `.shx`, `.prj` juntos), igual que en la sesión.

## <font color=green>11.1 Cargar los shapefiles</font>

In [ ]:
import geopandas as gpd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

RUTA_SHP = RUTA + "peru_shapes/"

regiones   = gpd.read_file("/content/peru_shapes/per_admbnda_adm1_ign_20200714.shp")   # ADM1: regiones
provincias = gpd.read_file("/content/peru_shapes/per_admbnda_adm2_ign_20200714.shp")   # ADM2: provincias
distritos  = gpd.read_file( "/content/peru_shapes/per_admbnda_adm3_ign_20200714.shp")   # ADM3: distritos

print("Regiones:", len(regiones), "| Provincias:", len(provincias), "| Distritos:", len(distritos))
print("CRS distritos:", distritos.crs)

## <font color=green>11.2 Llave de unión y auditoría del cruce (parte C del desafío)</font>

- **Nivel geográfico del proyecto:** distrito (× nivel educativo).
- **Llave:** `ubigeo_key` (6 dígitos: 2 región + 2 provincia + 2 distrito) de la matriz analítica ↔ `ADM3_PCODE` del shapefile, que es `"PE"` + el mismo ubigeo (ej. `PE010101`).
  Se usa el código y **no el nombre**, porque el shapefile viene sin tildes y los nombres del Padrón no coinciden exactamente.
- **Uso de la ubicación:** visualizar resultados (no entra al clustering).

Como el shapefile es de 2020 y el ubigeo del proyecto es más reciente, se audita qué distritos no cruzan antes de graficar.

In [ ]:
# Llave en el shapefile: quitamos el prefijo "PE" de ADM3_PCODE -> ubigeo de 6 dígitos.
distritos["ubigeo_key"] = distritos["ADM3_PCODE"].str[-6:]

# Nos aseguramos de que la llave del proyecto sea texto de 6 dígitos (ceros a la izquierda).
matriz_analitica["ubigeo_key"] = matriz_analitica["ubigeo_key"].astype(str).str.zfill(6)

claves_proyecto = set(matriz_analitica["ubigeo_key"])
claves_shape    = set(distritos["ubigeo_key"])

sin_poligono = sorted(claves_proyecto - claves_shape)   # están en el proyecto pero no en el mapa
sin_dato     = sorted(claves_shape - claves_proyecto)   # están en el mapa pero no en el proyecto

print(f"Distritos en el proyecto : {len(claves_proyecto)}")
print(f"Distritos en el shapefile: {len(claves_shape)}")
print(f"Cruzan                   : {len(claves_proyecto & claves_shape)}")
print(f"En el proyecto SIN polígono: {len(sin_poligono)}")
print(f"En el mapa SIN dato        : {len(sin_dato)}  (se pintarán en gris: no tienen deserción/matrícula reportada)")

if sin_poligono:
    display(matriz_analitica.loc[matriz_analitica["ubigeo_key"].isin(sin_poligono),
                                 ["ubigeo_key", "DPTO", "PROV", "DIST", "nivel_bin"]])

## <font color=green>11.3 Preparar una tabla por nivel y etiquetar los clusters</font>

Los números de cluster (0/1) son arbitrarios. Para que el mapa se lea sin ambigüedad, el cluster con **mayor `tasa_desercion_pct` promedio**
se etiqueta como *Salida alta* y el otro como *Salida baja* (coincide con el perfilamiento de las secciones 9 y 10).

In [ ]:
def etiquetar_clusters(col):
    """Devuelve un dict {cluster: 'Salida alta'/'Salida baja'} según la deserción promedio."""
    medias = matriz_analitica.groupby(col)["tasa_desercion_pct"].mean()
    alta = medias.idxmax()
    return {c: ("Salida alta" if c == alta else "Salida baja") for c in medias.index}

ORDEN = ["Salida baja", "Salida alta", "Sin dato"]
COLORES = ["#8FB8DE", "#C0392B", "#D9D9D9"]      # azul claro, rojo, gris
NIVELES = {0: "Primaria", 1: "Secundaria"}

for col in ["cluster_kmeans", "cluster_hier"]:
    mapa_etq = etiquetar_clusters(col)
    matriz_analitica["grupo_" + col.split("_")[1]] = matriz_analitica[col].map(mapa_etq)
    print(col, "->", mapa_etq)

def mapa_por_nivel(nivel_bin):
    """GeoDataFrame con TODOS los distritos del shapefile + los datos del proyecto para un nivel."""
    sub = matriz_analitica[matriz_analitica["nivel_bin"] == nivel_bin][
        ["ubigeo_key", "tasa_desercion_pct", "grupo_kmeans", "grupo_hier"]
    ]
    # GeoDataFrame a la IZQUIERDA para que el resultado siga siendo GeoDataFrame; how="left" conserva todos los polígonos.
    g = distritos[["ubigeo_key", "ADM1_ES", "ADM2_ES", "ADM3_ES", "geometry"]].merge(
        sub, on="ubigeo_key", how="left", validate="one_to_one")
    for c in ["grupo_kmeans", "grupo_hier"]:
        g[c] = pd.Categorical(g[c].fillna("Sin dato"), categories=ORDEN)
    return g

mapas = {n: mapa_por_nivel(b) for b, n in NIVELES.items()}
for n, g in mapas.items():
    print(n, "->", g["grupo_hier"].value_counts().to_dict())

## <font color=green>11.4 Mapa de los clusters por nivel educativo</font>

Se comparan Primaria y Secundaria lado a lado (misma escala de colores). Gris = el distrito no tiene dato en ese nivel
(por ejemplo, no hay secundaria o no reportó deserción).

In [ ]:
from matplotlib import patheffects

# Un punto dentro de cada departamento, calculado con el shapefile.
# No depende de coordenadas ni nombres escritos manualmente.
def poner_nombres_departamentos(ax, tamano=7):
    regiones_graf = regiones.to_crs(distritos.crs)
    for _, fila in regiones_graf.iterrows():
        nombre = str(fila["ADM1_ES"])
        if fila.geometry is None or fila.geometry.is_empty:
            continue
        punto = fila.geometry.representative_point()
        # Desplazamiento corto para evitar que las etiquetas de Lima y Callao se tapen.
        desplazamiento = (0, -9) if nombre.lower() == "callao" else (0, 0)
        txt = ax.annotate(
            nombre, xy=(punto.x, punto.y), xytext=desplazamiento,
            textcoords="offset points", ha="center", va="center",
            fontsize=tamano, fontweight="bold", color="#202020",
            zorder=15, annotation_clip=True
        )
        # Halo blanco: permite leer sobre los polígonos rojos y azules.
        txt.set_path_effects([
            patheffects.withStroke(linewidth=2.4, foreground="white")
        ])


def dibujar_clusters(col, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(15, 10))
    for ax, (nivel, g) in zip(axes, mapas.items()):
        g.plot(column=col, categorical=True, cmap=ListedColormap(COLORES),
               edgecolor="none", linewidth=0, ax=ax)
        regiones.to_crs(g.crs).boundary.plot(
            ax=ax, color="#242424", linewidth=0.65, zorder=5
        )
        poner_nombres_departamentos(ax, tamano=6.7)
        n_alta = (g[col] == "Salida alta").sum()
        ax.set_title(f"{nivel} — {n_alta} distritos con salida alta", fontsize=12)
        ax.set_axis_off()
    fig.legend(handles=[Patch(facecolor=c, label=o) for o, c in zip(ORDEN, COLORES)],
               loc="lower center", ncol=3, frameon=False)
    fig.suptitle(titulo, fontsize=14)
    plt.tight_layout(rect=[0, 0.04, 1, 0.96])
    plt.show()

dibujar_clusters("grupo_kmeans", "K-means (K = 2): salida escolar por distrito")

**Interpretación(K-means, K = 2):**

- La **salida alta es mucho más frecuente en secundaria**: 1 170 distritos frente a 345 en primaria.
- En **primaria** el patrón es geográficamente claro: la salida alta se concentra en la **selva** (norte y oriente), mientras que la sierra y la costa sur son mayormente de salida baja.
- En **secundaria** la selva también es de salida alta, pero además aparece en la **costa norte** y en muchos distritos de la sierra. El patrón es más fragmentado: distritos de salida alta y baja se alternan dentro de un mismo departamento.
- Los distritos en gris (sin dato) son pocos: 1 en primaria y 45 en secundaria.
- Los clusters se formaron solo con atributos; la ubicación se usa únicamente para dibujar.

## <font color=green>11.5 Mapa de la tasa de deserción (variable continua)</font>

Los clusters resumen cinco variables; aquí se mapea directamente `tasa_desercion_pct`. Se recorta la escala en el percentil 98
para que unos pocos valores extremos no aplasten el resto de colores.

In [ ]:
vmax = matriz_analitica["tasa_desercion_pct"].quantile(0.98)

fig, axes = plt.subplots(1, 2, figsize=(15, 10))
for ax, (nivel, g) in zip(axes, mapas.items()):
    g.plot(column="tasa_desercion_pct", cmap="rocket_r", vmin=0, vmax=vmax,
           edgecolor="none", ax=ax, legend=True,
           legend_kwds={"label": "Tasa de deserción (%)", "shrink": 0.5},
           missing_kwds={"color": "#D9D9D9"})
    regiones.to_crs(g.crs).boundary.plot(
        ax=ax, color="#242424", linewidth=0.65, zorder=5
    )
    poner_nombres_departamentos(ax, tamano=6.7)
    ax.set_title(nivel, fontsize=12)
    ax.set_axis_off()
fig.suptitle("Tasa de deserción 2023-2024 por distrito (gris = sin dato)", fontsize=14)
plt.tight_layout()
plt.show()


- En **primaria** las tasas son bajas en casi todo el país (tonos claros). Los valores altos se concentran en la selva norte y en la selva del sureste.
- En **secundaria** las tasas son más altas y más heterogéneas: además de la selva, hay manchas oscuras en la sierra norte y central y en la costa norte, y la variación entre distritos vecinos es grande.
- Este mapa muestra la variable continua; coincide con el de clusters (sección 11.4), donde los distritos de salida alta son los más oscuros aquí.

## <font color=green>11.6 Una región de interés </font>

Se elige la región con mayor porcentaje de distritos con *Salida alta* en **Secundaria** (entre las que tienen al menos 30 distritos con dato),
se muestran sus provincias, se selecciona la provincia con más distritos de salida alta y se mapean sus distritos.
Para analizar otra región basta cambiar `region_interes` (nombre sin tildes, ver `sorted(regiones["ADM1_ES"])`).

In [ ]:
matriz_analitica["region_shp"] = matriz_analitica["ubigeo_key"].map(
    distritos.set_index("ubigeo_key")["ADM1_ES"])

In [ ]:
sec = matriz_analitica[matriz_analitica["nivel_bin"] == 1]
res_reg = (sec.assign(alta=(sec["grupo_kmeans"] == "Salida alta"))
              .groupby("region_shp").agg(n=("alta", "size"), pct_alta=("alta", "mean")))
res_reg = res_reg[res_reg["n"] >= 30].sort_values("pct_alta", ascending=False)
display((100 * res_reg["pct_alta"]).round(1).head(8).to_frame("% distritos con salida alta (Secundaria)"))

region_interes = res_reg.index[0]          # <- cambiar aquí para otra región
print("Región de interés:", region_interes)

cod_reg = regiones.loc[regiones["ADM1_ES"] == region_interes, "ADM1_PCODE"].iloc[0]
prov_reg = provincias[provincias["ADM1_PCODE"] == cod_reg]

ax = prov_reg.plot(figsize=(7, 8), facecolor="white", edgecolor="black")
for _, row in prov_reg.iterrows():
    x, y = row["geometry"].representative_point().coords[0]   # punto garantizado dentro del polígono
    ax.annotate(row["ADM2_ES"], xy=(x, y), ha="center", fontsize=7, color="blue")
ax.set_title(f"Provincias de {region_interes}")
ax.set_axis_off()
plt.show()

In [ ]:
# Provincia con más distritos de Secundaria en "Salida alta" dentro de la región elegida.
g_sec = mapas["Secundaria"]
dist_reg = g_sec[g_sec["ADM1_ES"] == region_interes]
cuenta = dist_reg[dist_reg["grupo_kmeans"] == "Salida alta"].groupby("ADM2_ES").size().sort_values(ascending=False)
display(cuenta.head(5).to_frame("distritos con salida alta"))

provincia_interes = cuenta.index[0]        # <- cambiar aquí para otra provincia
dist_prov = dist_reg[dist_reg["ADM2_ES"] == provincia_interes]

ax = dist_prov.plot(column="grupo_kmeans", categorical=True, cmap=ListedColormap(COLORES),
                    edgecolor="white", linewidth=0.6, figsize=(8, 8))
for _, row in dist_prov.iterrows():
    x, y = row["geometry"].representative_point().coords[0]
    ax.annotate(row["ADM3_ES"], xy=(x, y), ha="center", fontsize=6, color="#222222")
ax.legend(handles=[Patch(facecolor=c, label=o) for o, c in zip(ORDEN, COLORES)],
          loc="lower left", frameon=False, fontsize=8)
ax.set_title(f"Distritos de {provincia_interes} ({region_interes}) — Secundaria, clustering Ward")
ax.set_axis_off()
plt.show()

display(dist_prov[["ADM3_ES", "grupo_kmeans", "tasa_desercion_pct"]]
        .sort_values("tasa_desercion_pct", ascending=False).round(2).head(10))


- **Lambayeque** tiene el mayor porcentaje de distritos de secundaria en *Salida alta* (92.1%), seguida de La Libertad (83.1%), Cajamarca (81.7%), Piura (81.5%) y Loreto (81.1%). Es decir, en esas regiones el problema no es de unos pocos distritos, sino casi generalizado. (Este ranking usa los clusters de Ward.)
- Dentro de Lambayeque, la provincia con más distritos de salida alta es **Chiclayo** (19 distritos), seguida de Lambayeque (10) y Ferreñafe (6).
- Los distritos con mayor tasa de deserción en esa provincia son **Monsefú (5.93%)**, **Eten Puerto (5.81%)** y **Reque (4.87%)**, que serían los primeros candidatos para una intervención de retención.

# **11. Geopandas**

---

## <font color=green>A. Proyecto</font>

Si su proyecto tiene componente espacial, indiquen cuál de estos enfoques usarían y por qué:

- clustering por atributos y luego mapa;
- clustering espacial (DBSCAN con coordenadas en metros);
- mapa de calor.

Si el proyecto no tiene componente espacial, no es necesario inventarlo.

In [ ]:
# DESAFÍO C: Aplicación del clustering al proyecto

# Enfoque seleccionado:
# Clustering por atributos y posterior visualización en un mapa.
# Este enfoque permite identificar dónde se ubican los distritos
# con salida escolar alta y baja, y comparar Primaria y Secundaria.


# Visualizar los clusters de K-means por nivel educativo
dibujar_clusters(
    "grupo_kmeans",
    "K-means: salida escolar por distrito"
)

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
for nombre, lab in [("K-means", matriz_analitica["cluster_kmeans"]),
                    ("Ward", matriz_analitica["cluster_hier"])]:
    print(nombre, "| Silhouette:", round(silhouette_score(X_db, lab), 3),
          "| Davies-Bouldin:", round(davies_bouldin_score(X_db, lab), 3))

Se compararon K-means (K = 2) y Ward (k = 2) con Silhouette y Davies-Bouldin sobre la misma matriz escalada (log1p + z-score). K-means obtuvo Silhouette 0.265 y Davies-Bouldin 1.465, frente a 0.238 y 1.637 de Ward. K-means tiene mayor Silhouette y menor Davies-Bouldin, por lo que sus grupos son algo más compactos y mejor separados. Por eso se elige K-means (K = 2) como solución final del proyecto, y Ward se conserva como contraste (ambos coinciden en el 84.6 % de los casos, así que la lectura de política es la misma). K-means además permite asignar distritos nuevos con predict, algo útil si el análisis se repite con datos de otros años.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 9))

for ax, (nivel, g) in zip(axes, mapas.items()):

    # Convertir los distritos en puntos
    puntos = g.copy()
    puntos["geometry"] = puntos.geometry.representative_point()

    # Dibujar el mapa del Perú
    regiones.plot(
        ax=ax,
        facecolor="white",
        edgecolor="gray",
        linewidth=0.5
    )

    # Distritos sin datos
    puntos[puntos["grupo_kmeans"] == "Sin dato"].plot(
        ax=ax, color="lightgray", markersize=4
    )

    # Distritos con salida escolar baja
    puntos[puntos["grupo_kmeans"] == "Salida baja"].plot(
        ax=ax, color="blue", markersize=8,
        label="Salida baja"
    )

    # Distritos con salida escolar alta
    puntos[puntos["grupo_kmeans"] == "Salida alta"].plot(
        ax=ax, color="red", markersize=8,
        label="Salida alta"
    )

    ax.set_title(f"Salida escolar - {nivel}")
    ax.set_axis_off()
    ax.legend()

plt.suptitle("Distribución espacial de la salida escolar por distrito")
plt.tight_layout()
plt.show()

**Lectura:** es el mismo resultado de la sección 11.4 pero con cada distrito representado por un punto, lo que evita que los distritos grandes de la selva dominen visualmente el mapa. Permite ver que en **secundaria** los puntos rojos (salida alta) son muchos más y están repartidos por todo el país, mientras que en **primaria** quedan concentrados en la selva.

# **12. Folium**

---

## <font color=green>12.1 Mapa coroplético interactivo de la tasa de deserción — Primaria y Secundaria</font>

Un mapa por nivel educativo, con la **misma escala de colores** (recortada en el percentil 98 de ambos niveles) para poder comparar a simple vista. Gris = distrito sin dato. Al pasar el mouse se ve región, provincia, distrito, tasa y cluster K-means. Cada mapa también se guarda como HTML.

In [ ]:
import folium
import json
from IPython.display import display

MAPA_BASE  = "OpenStreetMap.France"
crs_metros = distritos.estimate_utm_crs()

COL_CODIGO = "ADM3_PCODE"
COL_VALOR  = "tasa_desercion_pct"
NIVELES_121 = {0: "Primaria", 1: "Secundaria"}

# Misma escala para los dos niveles: recorte p98 sobre todos los datos (como en 11.5)
vmax = matriz_analitica[COL_VALOR].quantile(0.98)

# Geometrías simplificadas una sola vez (en metros) y devueltas a grados
base_geo = distritos.copy()
base_geo["geometry"] = base_geo.to_crs(crs_metros).simplify(500).to_crs(distritos.crs)


# Etiquetas permanentes de departamentos (calculadas a partir de las geometrías reales).
# Se colocan como una capa con opción de ocultar, para no saturar el mapa.
from branca.element import Element
from html import escape as html_escape


def agregar_nombres_departamentos(mapa, mostrar=True):
    if not hasattr(agregar_nombres_departamentos, "puntos"):
        regiones_etiquetas = (distritos[["ADM1_ES", "geometry"]]
                             .to_crs("EPSG:4326")
                             .dissolve(by="ADM1_ES"))
        proyeccion = regiones_etiquetas.estimate_utm_crs()
        centros = (regiones_etiquetas.to_crs(proyeccion)
                   .representative_point().to_crs("EPSG:4326"))
        agregar_nombres_departamentos.puntos = [
            (str(nombre), punto.y, punto.x)
            for nombre, punto in centros.items() if not punto.is_empty
        ]

    capa = folium.FeatureGroup(name="Nombres de departamentos", show=mostrar, overlay=True)
    for nombre, lat, lon in agregar_nombres_departamentos.puntos:
        etiqueta = ("<div style='font: bold 10px Arial,sans-serif;"
                    "color:#28323c;white-space:nowrap;text-align:center;"
                    "text-shadow:-1px -1px 2px white,1px 1px 2px white,"
                    "-1px 1px 2px white,1px -1px 2px white;"
                    "pointer-events:none;transform:translateX(-40%);'>"
                    + html_escape(nombre.title()) + "</div>")
        folium.Marker(
            location=[lat, lon],
            icon=folium.DivIcon(html=etiqueta, icon_size=(110, 16), icon_anchor=(55, 8)),
        ).add_to(capa)
    capa.add_to(mapa)
    return capa


def agregar_titulo_mapa(mapa, titulo, subtitulo="2023–2024"):
    contenido = ("<div style='position:fixed;top:12px;left:50%;"
                 "transform:translateX(-50%);z-index:9999;background:white;"
                 "padding:8px 14px;border:1px solid #ccc;border-radius:6px;"
                 "box-shadow:0 1px 5px #999;font:13px Arial,sans-serif;"
                 "text-align:center;pointer-events:none;max-width:65%;'>"
                 f"<b>{html_escape(titulo)}</b><br>"
                 f"<small>{html_escape(subtitulo)}</small></div>")
    mapa.get_root().html.add_child(Element(contenido))


# CONTORNOS DE DEPARTAMENTOS: bordes de mayor grosor que los distritales.
# Se calculan una sola vez y se reutilizan en todos los mapas.
def agregar_limites_departamentales(mapa, mostrar=True):
    if not hasattr(agregar_limites_departamentales, "geojson"):
        departamentos = (distritos[["ADM1_ES", "geometry"]]
                         .to_crs("EPSG:4326")
                         .dissolve(by="ADM1_ES"))
        crs_m = departamentos.estimate_utm_crs()
        departamentos["geometry"] = (departamentos.to_crs(crs_m)
                                     .geometry.simplify(1200, preserve_topology=True)
                                     .to_crs("EPSG:4326"))
        agregar_limites_departamentales.geojson = json.loads(departamentos.to_json())

    capa = folium.FeatureGroup(
        name="Límites departamentales", show=mostrar, overlay=True
    )
    folium.GeoJson(
        agregar_limites_departamentales.geojson,
        name="Contornos departamentales",
        style_function=lambda feature: {
            "color": "#353535", "weight": 2.0,
            "opacity": 0.85, "fillOpacity": 0,
        },
        control=False,
    ).add_to(capa)
    capa.add_to(mapa)
    return capa

mapas_121 = {}
for NIVEL, nombre in NIVELES_121.items():
    tabla = matriz_analitica.loc[matriz_analitica["nivel_bin"] == NIVEL,
                                 ["ubigeo_key", COL_VALOR, "grupo_kmeans"]].copy()
    tabla[COL_CODIGO] = "PE" + tabla["ubigeo_key"]     # ubigeo -> código del shapefile
    tabla = tabla.drop(columns="ubigeo_key")           # evita ubigeo_key_x / _y en el merge

    mapa_tabla = base_geo.merge(tabla, on=COL_CODIGO, how="left", validate="one_to_one")

    m = mapa_tabla.explore(
        column=COL_VALOR, cmap="OrRd", vmin=0, vmax=vmax, legend=True,
        tooltip=["ADM1_ES", "ADM2_ES", "ADM3_ES", COL_VALOR, "grupo_kmeans"],
        tiles=MAPA_BASE,
        missing_kwds={"color": "lightgray", "label": "sin dato"},
        style_kwds={"color": "gray", "weight": 0.3}
    )
    agregar_limites_departamentales(m)
    agregar_nombres_departamentos(m)
    agregar_titulo_mapa(m, f"Tasa de deserción escolar — {nombre}")
    folium.LayerControl(collapsed=True).add_to(m)
    archivo = f"mapa_desercion_distritos_{nombre.lower()}.html"
    m.save(archivo)
    mapas_121[nombre] = m
    print(f"{nombre}: {mapa_tabla[COL_VALOR].notna().sum()} distritos con dato | guardado en {archivo}")
    display(m)

**Interpretación:** los dos mapas comparten escala, así que se pueden comparar directamente. En primaria casi todo el país está en tonos claros y los distritos oscuros se concentran en la selva. En secundaria el mapa es más oscuro y más heterogéneo: hay distritos de tasa alta también en la sierra y la costa norte, y distritos vecinos con tasas muy distintas. Las zonas en gris son distritos sin dato en ese nivel.

## <font color=green>12.2 Preparar datos para los mapas con capas</font>

Se prepara **una sola vez** el mapa de distritos: geometrías simplificadas (1 km) en grados, el punto representativo de cada distrito (latitud/longitud) y una tabla por nivel con la tasa, los clusters y el número de desertores. Folium dibuja en **grados (EPSG:4326)**; la simplificación se hace en metros.

Los mapas muestran la solución final de **K-means**, seleccionada en la sección 11.A (mayor Silhouette y menor Davies-Bouldin). Los tooltips muestran exclusivamente este cluster, sin Ward.

In [ ]:
import json
from folium.plugins import HeatMap
from branca.colormap import LinearColormap

MAPA_PROY = "Esri.WorldGrayCanvas"   # fondo gris neutro (Esri, como en la clase): los datos resaltan más
COL_CLUSTER = "grupo_kmeans"         # Solución final utilizada en todos los mapas
NOMBRE_CLUSTER = "K-means"

# Geometrías en grados -> simplificar en metros -> volver a grados
geo_base = distritos[["ubigeo_key", "ADM1_ES", "ADM2_ES", "ADM3_ES", "geometry"]].to_crs("EPSG:4326")
geo_m = geo_base.to_crs(geo_base.estimate_utm_crs())
pt = geo_m.geometry.representative_point().to_crs("EPSG:4326")      # punto garantizado dentro del distrito
geo_base["lat"] = pt.y.values
geo_base["lon"] = pt.x.values
geo_base["geometry"] = geo_m.geometry.simplify(1000).to_crs("EPSG:4326")

def tabla_nivel(nivel_bin):
    """Todos los distritos del shapefile + datos del proyecto para un nivel (Primaria=0, Secundaria=1)."""
    sub = matriz_analitica.loc[matriz_analitica["nivel_bin"] == nivel_bin,
                               ["ubigeo_key", "tasa_desercion_pct", "grupo_kmeans", "grupo_hier"]]
    des = base_final.loc[base_final["dsc_nivel"] == NIVELES[nivel_bin],
                         ["ubigeo_key", "n_desertores", "n_matriculados_base_desercion"]]
    g = (geo_base.merge(sub, on="ubigeo_key", how="left", validate="one_to_one")
                 .merge(des, on="ubigeo_key", how="left", validate="one_to_one"))
    g["grupo_hier"] = g["grupo_hier"].fillna("Sin dato")
    g["grupo_kmeans"] = g["grupo_kmeans"].fillna("Sin dato")
    g["tasa_txt"] = g["tasa_desercion_pct"].round(2).astype(str).replace("nan", "sin dato")
    g["desertores_txt"] = g["n_desertores"].astype("Int64").astype(str).replace("<NA>", "sin dato")
    return g

tablas = {n: tabla_nivel(b) for b, n in NIVELES.items()}

# Escala de color común para ambos niveles (recorte p98, igual que en 11.5)
vmax_tasa = float(matriz_analitica["tasa_desercion_pct"].quantile(0.98))
escala_tasa = LinearColormap(["#fff7ec", "#fdbb84", "#e34a33", "#7f0000"], vmin=0, vmax=vmax_tasa,
                             caption="Tasa de deserción 2023-2024 (%)")
COLOR_GRUPO = {"Salida alta": "#C0392B", "Salida baja": "#8FB8DE", "Sin dato": "#D9D9D9"}

COLS_TOOLTIP = ["ADM1_ES", "ADM2_ES", "ADM3_ES", "tasa_txt", "desertores_txt", "grupo_kmeans"]
ALIAS_TOOLTIP = ["Región", "Provincia", "Distrito", "Tasa (%)", "Desertores", "Cluster K-means"]

def capa_tasa(nivel, show=True):
    g = tablas[nivel]
    datos = json.loads(g[COLS_TOOLTIP + ["tasa_desercion_pct", "geometry"]].to_json())
    def estilo(f):
        v = f["properties"]["tasa_desercion_pct"]
        if v is None:
            return {"fillColor": "#D9D9D9", "color": "gray", "weight": 0.3, "fillOpacity": 0.6}
        return {"fillColor": escala_tasa(min(v, vmax_tasa)), "color": "gray", "weight": 0.3, "fillOpacity": 0.85}
    return folium.GeoJson(datos, name=f"Tasa de deserción — {nivel}", style_function=estilo, show=show,
                          tooltip=folium.GeoJsonTooltip(fields=COLS_TOOLTIP, aliases=ALIAS_TOOLTIP, sticky=True))

def capa_clusters(nivel, col=None, show=True):
    col = col or COL_CLUSTER
    g = tablas[nivel]
    datos = json.loads(g[COLS_TOOLTIP + ["geometry"]].to_json())
    def estilo(f):
        return {"fillColor": COLOR_GRUPO[f["properties"][col]], "color": "gray", "weight": 0.3, "fillOpacity": 0.85}
    nombre = "K-means" if col == "grupo_kmeans" else "Ward"
    return folium.GeoJson(datos, name=f"Clusters {nombre} — {nivel}", style_function=estilo, show=show,
                          tooltip=folium.GeoJsonTooltip(fields=COLS_TOOLTIP, aliases=ALIAS_TOOLTIP, sticky=True))

def leyenda_clusters():
    filas = "".join(f'<div><span style="background:{c};width:12px;height:12px;display:inline-block;'
                    f'margin-right:6px;border:1px solid #888"></span>{k}</div>' for k, c in COLOR_GRUPO.items())
    html = ('<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:8px 12px;'
            'border:1px solid #999;border-radius:4px;font-size:12px"><b>Clusters</b>' + filas + '</div>')
    return folium.Element(html)

CENTRO_PERU = [-9.2, -75.0]
print({n: len(t) for n, t in tablas.items()}, "distritos en el mapa | clusters:", NOMBRE_CLUSTER)

## <font color=green>12.3 Mapas de clusters — Primaria y Secundaria</font>

Cada distrito se pinta según su cluster: **Salida alta** (rojo), **Salida baja** (azul), **Sin dato** (gris). Al pasar el mouse se ve la región, la provincia, la tasa y los desertores.

In [ ]:
for nivel in ["Primaria", "Secundaria"]:
    m_clu = folium.Map(location=CENTRO_PERU, zoom_start=5, tiles=MAPA_PROY)
    capa_clusters(nivel).add_to(m_clu)
    m_clu.add_child(leyenda_clusters())
    agregar_limites_departamentales(m_clu)
    agregar_nombres_departamentos(m_clu)
    agregar_titulo_mapa(m_clu, f"Clusters K-means — {nivel}")
    folium.LayerControl(collapsed=True).add_to(m_clu)
    m_clu.save(f"mapa_clusters_{nivel.lower()}.html")
    print(f"{nivel}: clusters {NOMBRE_CLUSTER}")
    display(m_clu)

**Interpretación:** En primaria, los distritos de salida alta (rojo) están sobre todo en la selva; en secundaria son mucho más numerosos y también aparecen en la costa norte y en la sierra. El tooltip permite identificar el distrito concreto y ver su tasa, útil para que una UGEL ubique los distritos de su jurisdicción.

## <font color=green>12.4 Mapa de calor de desertores (por distrito)</font>

El mapa de calor no añade nombres de departamentos: solo se dibujan sus contornos para no duplicar las etiquetas del mapa base. El proyecto no tiene coordenadas por evento, así que cada distrito aporta **un punto** (su punto representativo) con peso igual a la raíz cuadrada del **número de desertores**. La raíz evita que unos pocos distritos de Lima aplasten el resto. Ojo: este mapa muestra *dónde se concentra el número de estudiantes que salen*, que no es lo mismo que la *tasa* (los distritos grandes pesan más aunque su tasa sea baja).

In [ ]:
m_calor = folium.Map(location=CENTRO_PERU, zoom_start=5, tiles=MAPA_PROY)
for nivel, mostrar in [("Secundaria", True), ("Primaria", False)]:
    g = tablas[nivel].dropna(subset=["n_desertores"])
    pesos = np.sqrt(g["n_desertores"].astype(float))
    capa = folium.FeatureGroup(name=f"Calor de desertores — {nivel}", show=mostrar)
    HeatMap(list(zip(g["lat"], g["lon"], pesos)), radius=14, blur=18, min_opacity=0.3,
            gradient={0.2: "#fdbb84", 0.4: "#fc8d59", 0.6: "#e34a33", 0.8: "#b30000", 1.0: "#7f0000"}).add_to(capa)
    capa.add_to(m_calor)
agregar_limites_departamentales(m_calor)
# Sin etiquetas añadidas: el mapa base ya muestra algunos nombres
agregar_titulo_mapa(m_calor, "Concentración de desertores por distrito")
folium.LayerControl(collapsed=False).add_to(m_calor)
m_calor.save("mapa_calor_desertores.html")
m_calor

**Interpretación:** el calor marca dónde se concentra el **número** de estudiantes que dejan de matricularse, no la tasa. Por eso las manchas más intensas coinciden con las zonas más pobladas, y no necesariamente con los distritos de mayor tasa (selva). Sirve para priorizar volumen de estudiantes; el mapa de clusters sirve para priorizar intensidad del problema.

## <font color=green>12.5 Mapa final con capas</font>

Mapa interactivo con selector de capas:

- tasa de deserción en **Secundaria** y **Primaria** (misma escala de color);
- clusters **K-means** de Secundaria y Primaria; al pasar el mouse se muestra solo K-means, sin Ward;
- calor por número de desertores de Secundaria;
- **Top 10 distritos con mayor tasa de deserción**, por nivel, con puntos amarillos (por defecto, Secundaria);
- **Top 10 con mayor brecha Secundaria − Primaria**, como capa adicional opcional de puntos turquesas, oculta por defecto.

Para interpretar los puntos amarillos se debe activar la capa de tasa del **mismo nivel**. Se usa el filtro de al menos **60 matriculados en la base** para reducir tasas inestables en distritos pequeños. El mapa se guarda como HTML.

**Departamentos:** sus límites están delineados en gris oscuro; los nombres aparecen en una capa independiente que se puede desactivar. Los distritos se identifican al pasar el cursor. **Top 10:** ambas capas usan puntos amarillos, pero se activa solo secundaria al abrir para evitar superposiciones. Activa primaria y desactiva secundaria si deseas analizar primaria.


In [ ]:
m_final = folium.Map(location=CENTRO_PERU, zoom_start=5, tiles=MAPA_PROY)

# Capas de polígonos (solo la primera encendida: dos capas opacas encendidas a la vez se tapan)
capa_tasa("Secundaria", show=True).add_to(m_final)
capa_tasa("Primaria", show=False).add_to(m_final)
capa_clusters("Secundaria", show=False).add_to(m_final)
capa_clusters("Primaria", show=False).add_to(m_final)

# Capas de calor independientes para Secundaria y Primaria.
# Ambas comienzan ocultas; actívalas desde el control de capas.
for nivel in ["Secundaria", "Primaria"]:
    g_calor = tablas[nivel].dropna(subset=["lat", "lon", "n_desertores"]).copy()
    g_calor = g_calor.loc[g_calor["n_desertores"].astype(float) >= 0]
    pesos = np.sqrt(g_calor["n_desertores"].astype(float))

    capa_calor = folium.FeatureGroup(
        name=f"Calor de desertores — {nivel}", show=False
    )
    HeatMap(
        list(zip(g_calor["lat"], g_calor["lon"], pesos)),
        radius=14,
        blur=18,
        min_opacity=0.3,
        gradient={
            0.2: "#fdbb84", 0.4: "#fc8d59", 0.6: "#e34a33",
            0.8: "#b30000", 1.0: "#7f0000"
        }
    ).add_to(capa_calor)
    capa_calor.add_to(m_final)

# Top 10 distritos con mayor TASA de deserción (no con mayor brecha).
# Mismo filtro que la sección 11.2: denominador >= 60 para evitar tasas inestables.
# Cada nivel tiene sus propios 10 distritos, con puntos situados dentro del polígono.
for nivel in ["Secundaria", "Primaria"]:
    candidatos = tablas[nivel].dropna(subset=["tasa_desercion_pct", "lat", "lon"]).copy()
    candidatos = candidatos.loc[candidatos["n_matriculados_base_desercion"] >= 60]
    top10 = candidatos.nlargest(10, "tasa_desercion_pct")

    capa_top10 = folium.FeatureGroup(
        name=f"🟡 Top 10 mayor tasa — {nivel}",
        show=(nivel == "Secundaria")
    )
    for puesto, (_, f) in enumerate(top10.iterrows(), start=1):
        # Información completa directamente al pasar el mouse por el punto amarillo.
        texto = (f"<b>Top {puesto} — {nivel}</b><br>"
                 f"<b>Región:</b> {f['ADM1_ES']}<br>"
                 f"<b>Provincia:</b> {f['ADM2_ES']}<br>"
                 f"<b>Distrito:</b> {f['ADM3_ES']}<br>"
                 f"<b>Tasa de deserción:</b> {f['tasa_desercion_pct']:.2f}%<br>"
                 f"<b>Desertores:</b> {f['desertores_txt']}<br>"
                 f"<b>Cluster K-means:</b> {f['grupo_kmeans']}")
        folium.CircleMarker(
            location=[f["lat"], f["lon"]], radius=8,
            color="#222222", weight=2, fill=True,
            fill_color="#FFD166", fill_opacity=1,
            tooltip=folium.Tooltip(texto, sticky=True),
            popup=folium.Popup(texto, max_width=300)
        ).add_to(capa_top10)
    capa_top10.add_to(m_final)
    print(f"Top 10 tasa de deserción — {nivel} (puntos amarillos)")
    display(top10[["ADM1_ES", "ADM2_ES", "ADM3_ES", "tasa_desercion_pct"]])

# Capa opcional: Top 10 BRECHA entre Secundaria y Primaria (independiente).
# Se calcula aquí mismo, sin depender de la variable `comp` de otras celdas.
comparacion = (
    tablas["Primaria"][["ubigeo_key", "ADM1_ES", "ADM2_ES", "ADM3_ES", "lat", "lon",
                        "tasa_desercion_pct"]]
    .rename(columns={"tasa_desercion_pct": "tasa_Primaria"})
    .merge(
        tablas["Secundaria"][["ubigeo_key", "tasa_desercion_pct"]]
        .rename(columns={"tasa_desercion_pct": "tasa_Secundaria"}),
        on="ubigeo_key", how="inner", validate="one_to_one"
    )
    .dropna(subset=["tasa_Primaria", "tasa_Secundaria", "lat", "lon"])
)
comparacion["brecha"] = comparacion["tasa_Secundaria"] - comparacion["tasa_Primaria"]
top_brecha = comparacion.nlargest(10, "brecha")
capa_brecha = folium.FeatureGroup(
    name="Top 10 mayor brecha Secundaria − Primaria (turquesa)", show=False
)
for puesto, (_, f) in enumerate(top_brecha.iterrows(), start=1):
    texto = (f"<b>#{puesto} — {f['ADM3_ES']}</b> ({f['ADM1_ES']})<br>"
             f"Provincia: {f['ADM2_ES']}<br>"
             f"Primaria: {f['tasa_Primaria']:.2f}%<br>"
             f"Secundaria: {f['tasa_Secundaria']:.2f}%<br>"
             f"Brecha: {f['brecha']:+.2f} pp")
    folium.CircleMarker(
        location=[f["lat"], f["lon"]], radius=7, color="black", weight=1.5,
        fill=True, fill_color="#52D6C5", fill_opacity=0.95,
        tooltip=f"Brecha #{puesto} · {f['ADM3_ES']}",
        popup=folium.Popup(texto, max_width=250)
    ).add_to(capa_brecha)
capa_brecha.add_to(m_final)

# Límites y etiquetas departamentales, cada uno en su propia capa.
agregar_limites_departamentales(m_final)
agregar_nombres_departamentos(m_final)
agregar_titulo_mapa(
    m_final,
    "Deserción escolar por distrito — Perú",
    "2023–2024 · Elige tasa y Top 10 del mismo nivel en Capas"
)

escala_tasa.add_to(m_final)
m_final.add_child(leyenda_clusters())
folium.LayerControl(collapsed=False).add_to(m_final)

m_final.save("mapa_proyecto_desercion.html")
print("Guardado: mapa_proyecto_desercion.html")
m_final

**Interpretación:** las áreas rojas representan tasas de deserción elevadas (la escala satura por encima del percentil 98), mientras que los puntos amarillos identifican los **10 distritos con mayor tasa** del nivel seleccionado, aplicando un mínimo de 60 matriculados en la base. Un distrito puede verse rojo oscuro sin formar parte del Top 10. Los puntos turquesas opcionales representan la **brecha** Secundaria − Primaria, que es un indicador diferente. La capa de calor muestra el volumen de desertores, no la tasa; las capas de clusters representan exclusivamente K-means.

**Limitaciones:** la ubicación se usa solo para visualizar; ningún clustering usó coordenadas. Los distritos sin datos se indican en gris. El recorte de color en p98 agrupa visualmente tasas extremas con un mismo tono.


In [ ]:
from google.colab import files

for nombre in ["primaria", "secundaria"]:
    files.download(f"mapa_desercion_distritos_{nombre}.html")